In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
from cnmf import Preprocess, cNMF
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, to_hex
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram, set_link_color_palette
from scipy.spatial import procrustes
from scipy.spatial.distance import squareform, pdist
from scipy.stats import spearmanr, chi2, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from statsmodels.stats.multitest import multipletests
from typing import Optional, Union, Sequence, Dict, Tuple
import anndata as ad
import glob
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scanpy as sc
import seaborn as sns
import umap
import warnings

warnings.filterwarnings("ignore")

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42

os.makedirs("figures", exist_ok=True)
os.makedirs("results/cnmf", exist_ok=True)
os.makedirs("results/cpdb/imt", exist_ok=True)


In [ ]:
T_all = sc.read_h5ad( 'T.h5ad')


In [ ]:
T_all_tumor= T_all[T_all.obs['Sample_Type'].isin(['T'])].copy()


In [ ]:
meta_color_map = {
    "Child": "#fdae6b",
    "Adult": "#9ecae1",
    'T': '#ffc89d',
    'B': '#f2a2af',
    '10X5': '#57CBE1',
    '10X3': '#B8DFC6',
    "NB": "#e6550d",
    "HB":  "#fd8d3c",
    "WT":  "#ffed6f",
    'PB': "#ffffb3" ,
    "MB":  "#e7969c",
    'pLGG':  "#d6616b",
    'pHGG':  "#ad494a",
    "BRCA":"#3182bd",
    'CRC':"#6baed6",
    'PC':"#8ca252",
    "LC":"#b5cf6b",
    "HCC":"#756BB1",
    'RC':"#bebada",
    'aHCC':"#a55194",
}


# Fig5a

In [ ]:
def prepare_composition_matrix(
    df,
    name="composition matrix",
    drop_all_zero=True,
):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")

    x = df.copy()

    x.index = x.index.astype(str)
    x.index.name = "Sample_ID"
    x.columns.name = None

    if x.index.duplicated().any():
        duplicated_samples = (
            x.index[x.index.duplicated()]
            .unique()
            .tolist()
        )
        raise ValueError(
            f"{name} contains duplicated sample IDs: "
            f"{duplicated_samples[:10]}"
        )

    x = x.apply(pd.to_numeric, errors="raise").astype(float)

    if x.isna().any().any():
        raise ValueError(f"{name} contains missing values.")

    if np.isinf(x.to_numpy()).any():
        raise ValueError(f"{name} contains infinite values.")

    if (x < 0).any().any():
        raise ValueError(f"{name} contains negative values.")

    if drop_all_zero:
        all_zero_features = x.columns[x.sum(axis=0) == 0]

        if len(all_zero_features) > 0:
            print(
                f"{name}: removing {len(all_zero_features)} "
                "all-zero features:"
            )
            print(all_zero_features.tolist())

            x = x.drop(columns=all_zero_features)

    if x.shape[1] < 2:
        raise ValueError(
            f"{name} must contain at least two non-zero features."
        )

    row_sums = x.sum(axis=1)

    if (row_sums <= 0).any():
        bad_samples = row_sums.index[row_sums <= 0].tolist()

        raise ValueError(
            f"{name} contains samples with row sum <= 0: "
            f"{bad_samples[:10]}"
        )

    x = x.div(row_sums, axis=0)

    return x

def multiplicative_zero_replacement(
    df,
    delta=None,
):
    x = df.to_numpy(dtype=float).copy()

    n_samples, n_features = x.shape

    if delta is None:
        delta = 0.65 / (n_features ** 2)

    if delta <= 0:
        raise ValueError("delta must be greater than zero.")

    for i in range(n_samples):
        zero_mask = x[i] == 0
        n_zero = int(zero_mask.sum())

        if n_zero == 0:
            continue

        nonzero_mask = ~zero_mask
        nonzero_sum = x[i, nonzero_mask].sum()

        if nonzero_sum <= 0:
            raise ValueError(
                f"Sample {df.index[i]!r} contains no positive values."
            )

        delta_i = min(
            delta,
            0.5 / n_zero,
        )

        zero_mass = n_zero * delta_i
        remaining_mass = 1.0 - zero_mass

        x[i, zero_mask] = delta_i

        x[i, nonzero_mask] = (
            x[i, nonzero_mask]
            / nonzero_sum
            * remaining_mass
        )

    x = x / x.sum(axis=1, keepdims=True)

    return pd.DataFrame(
        x,
        index=df.index,
        columns=df.columns,
    )

def clr_transform(
    df,
    delta=None,
):
    x_replaced = multiplicative_zero_replacement(
        df,
        delta=delta,
    )

    log_x = np.log(x_replaced)

    clr_x = log_x.sub(
        log_x.mean(axis=1),
        axis=0,
    )

    return clr_x

def transform_sample_composition(
    comp_raw,
    transform="clr",
    delta=None,
    center_features=True,
    scale_features=False,
):
    comp_normalized = prepare_composition_matrix(
        comp_raw,
        name="Sample composition matrix",
    )

    transform = transform.lower()

    if transform == "raw":
        comp_feature_space = comp_normalized.copy()

    elif transform == "hellinger":
        comp_feature_space = np.sqrt(comp_normalized)

    elif transform == "clr":
        comp_feature_space = clr_transform(
            comp_normalized,
            delta=delta,
        )

    else:
        raise ValueError(
            "transform must be one of: "
            "'raw', 'hellinger', or 'clr'"
        )

    comp_transformed = comp_feature_space.copy()

    if center_features:
        comp_transformed = (
            comp_transformed
            - comp_transformed.mean(axis=0)
        )

    if scale_features:
        feature_sd = comp_transformed.std(
            axis=0,
            ddof=0,
        )

        variable_features = feature_sd > 1e-12
        removed_features = feature_sd.index[
            ~variable_features
        ]

        if len(removed_features) > 0:
            print(
                f"Removing {len(removed_features)} "
                "zero-variance features:"
            )
            print(removed_features.tolist())

        comp_transformed = comp_transformed.loc[
            :,
            variable_features,
        ]

        feature_sd = feature_sd.loc[
            variable_features
        ]

        comp_transformed = comp_transformed.div(
            feature_sd,
            axis=1,
        )

    return (
        comp_normalized,
        comp_feature_space,
        comp_transformed,
    )

def run_sample_composition_tripleplot(
    adata,
    sample_col="Sample_ID",
    subset_col="T_sub_level_3",
    meta_cols=("Period_merged", "Disease"),
    min_cells_per_sample=20,
    transform="clr",
    delta=None,
    center_features=True,
    scale=False,
    n_pcs=5,
    umap_n_neighbors=3,
    umap_min_dist=0.3,
    random_state=0,
    color_col="Disease",
    style_col="Period_merged",
    figsize=(18, 5),
    point_size=80,
    save=None,
):

    required_obs_cols = {
        sample_col,
        subset_col,
        *meta_cols,
        color_col,
        style_col,
    }

    missing_obs_cols = [
        col
        for col in required_obs_cols
        if col not in adata.obs.columns
    ]

    if missing_obs_cols:
        raise KeyError(
            f"Missing columns in adata.obs: {missing_obs_cols}"
        )

    obs = (
        adata.obs[[sample_col, subset_col]]
        .dropna(subset=[sample_col, subset_col])
        .copy()
    )

    obs[sample_col] = obs[sample_col].astype(str)

    comp_counts = pd.crosstab(
        obs[sample_col],
        obs[subset_col],
    )

    sample_sizes = comp_counts.sum(axis=1)

    keep_samples = sample_sizes[
        sample_sizes >= min_cells_per_sample
    ].index

    comp_counts = comp_counts.loc[keep_samples]
    sample_sizes = sample_sizes.loc[keep_samples]

    if comp_counts.shape[0] < 3:
        raise ValueError(
            "Fewer than three samples remain after filtering."
        )

    (
        comp_raw,
        comp_feature_space,
        comp_transformed,
    ) = transform_sample_composition(
        comp_raw=comp_counts,
        transform=transform,
        delta=delta,
        center_features=center_features,
        scale_features=scale,
    )

    meta_cols_all = list(
        dict.fromkeys(
            [
                sample_col,
                *meta_cols,
                color_col,
                style_col,
            ]
        )
    )

    meta = adata.obs[meta_cols_all].copy()
    meta[sample_col] = meta[sample_col].astype(str)

    meta_unique = meta.drop_duplicates()

    duplicate_counts = (
        meta_unique
        .groupby(sample_col, observed=True)
        .size()
    )

    if (duplicate_counts > 1).any():
        bad_samples = (
            duplicate_counts[
                duplicate_counts > 1
            ]
            .index
            .tolist()
        )

        raise ValueError(
            f"Some {sample_col} values map to multiple "
            f"metadata rows, e.g. {bad_samples[:10]}"
        )

    sample_meta = (
        meta_unique
        .drop_duplicates(subset=sample_col)
        .set_index(sample_col)
        .reindex(comp_transformed.index)
    )

    if sample_meta[color_col].isna().any():
        bad_samples = sample_meta.index[
            sample_meta[color_col].isna()
        ].tolist()

        raise ValueError(
            f"Missing {color_col} metadata for samples: "
            f"{bad_samples[:10]}"
        )

    if sample_meta[style_col].isna().any():
        bad_samples = sample_meta.index[
            sample_meta[style_col].isna()
        ].tolist()

        raise ValueError(
            f"Missing {style_col} metadata for samples: "
            f"{bad_samples[:10]}"
        )

    pca_full = PCA(
        svd_solver="full",
    )

    X_pca_full = pca_full.fit_transform(
        comp_transformed.to_numpy(dtype=float)
    )

    evr = pca_full.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    if X_pca_full.shape[1] < 2:
        raise ValueError(
            "PCA returned fewer than two components."
        )

    emb_pca = pd.DataFrame(
        X_pca_full[:, :2],
        index=comp_transformed.index,
        columns=["PC1", "PC2"],
    )

    max_n_pcs = min(
        comp_transformed.shape[0] - 1,
        comp_transformed.shape[1],
    )

    n_pcs_used = min(
        n_pcs,
        max_n_pcs,
    )

    if n_pcs_used < 2:
        raise ValueError(
            "At least two PCs are required for UMAP."
        )

    if n_pcs_used != n_pcs:
        print(
            f"n_pcs was reduced from {n_pcs} "
            f"to {n_pcs_used}."
        )

    pca_umap = PCA(
        n_components=n_pcs_used,
        svd_solver="full",
    )

    X_pca = pca_umap.fit_transform(
        comp_transformed.to_numpy(dtype=float)
    )

    n_neighbors_used = min(
        umap_n_neighbors,
        comp_transformed.shape[0] - 1,
    )

    n_neighbors_used = max(
        2,
        n_neighbors_used,
    )

    if n_neighbors_used != umap_n_neighbors:
        print(
            f"umap_n_neighbors was adjusted from "
            f"{umap_n_neighbors} to {n_neighbors_used}."
        )

    umap_model = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors_used,
        min_dist=umap_min_dist,
        metric="euclidean",
        random_state=random_state,
    )

    X_umap = umap_model.fit_transform(
        X_pca
    )

    emb_umap = pd.DataFrame(
        X_umap,
        index=comp_transformed.index,
        columns=["UMAP1", "UMAP2"],
    )

    plot_df_pca = emb_pca.join(
        sample_meta,
        how="left",
    )

    plot_df_umap = emb_umap.join(
        sample_meta,
        how="left",
    )

    pc1_var = evr[0] * 100
    pc2_var = evr[1] * 100

    fig, axes = plt.subplots(
        1,
        3,
        figsize=figsize,
    )

    sns.scatterplot(
        data=plot_df_pca,
        x="PC1",
        y="PC2",
        hue=color_col,
        style=style_col,
        s=point_size,
        linewidth=0,
        ax=axes[0],
    )

    axes[0].set_title(
        f"PCA ({transform.upper()})"
    )

    axes[0].set_xlabel(
        f"PC1 ({pc1_var:.1f}%)"
    )

    axes[0].set_ylabel(
        f"PC2 ({pc2_var:.1f}%)"
    )

    axes[1].plot(
        range(1, len(evr) + 1),
        evr,
        marker="o",
    )

    axes[1].set_xlabel(
        "Principal component"
    )

    axes[1].set_ylabel(
        "Explained variance ratio"
    )

    axes[1].set_title(
        f"Scree plot ({transform.upper()})"
    )

    sns.scatterplot(
        data=plot_df_umap,
        x="UMAP1",
        y="UMAP2",
        hue=color_col,
        style=style_col,
        s=point_size,
        linewidth=0,
        legend=False,
        ax=axes[2],
    )

    axes[2].set_title(
        f"UMAP ({transform.upper()}, top {n_pcs_used} PCs)"
    )

    axes[2].set_xlabel("UMAP1")
    axes[2].set_ylabel("UMAP2")

    for ax in axes:
        for spine in ax.spines.values():
            spine.set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()

    if len(handles) > 0:
        axes[0].legend(
            handles=handles,
            labels=labels,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
        )

    plt.tight_layout()

    if save is not None:
        plt.savefig(
            save,
            bbox_inches="tight",
            transparent=True,
        )

    plt.show()

    return {
        "comp_counts": comp_counts,
        "comp_raw": comp_raw,
        "comp_feature_space": comp_feature_space,
        "comp_transformed": comp_transformed,
        "sample_sizes": sample_sizes,
        "sample_meta": sample_meta,
        "plot_df_pca": plot_df_pca,
        "plot_df_umap": plot_df_umap,
        "emb_pca": emb_pca,
        "emb_umap": emb_umap,
        "explained_variance_ratio": evr,
        "cumulative_explained_variance": cum_evr,
        "n_pcs_used": n_pcs_used,
        "umap_n_neighbors_used": n_neighbors_used,
        "pca_full_model": pca_full,
        "pca_umap_model": pca_umap,
        "umap_model": umap_model,
    }


In [ ]:
def plot_sample_embedding(
    emb,
    sample_meta=None,
    color_col=None,
    style_col=None,
    color_map=None,
    figsize=(6.8, 5),
    point_size=80,
    title=None,
    annotate=False,
    legend_outside=True,
    legend_anchor=(1.02, 1.0),
    main_width_ratio=0.78,
    save=None,
    dpi=300,
):
    plot_df = emb.copy()

    if sample_meta is not None:
        plot_df = plot_df.join(sample_meta, how="left")

    xcol, ycol = plot_df.columns[:2]

    hue_order = None
    if color_col is not None and color_map is not None:
        present_levels = pd.unique(plot_df[color_col].dropna())
        hue_order = [k for k in color_map.keys() if k in present_levels]

        missing_levels = [x for x in present_levels if x not in color_map]
        hue_order = hue_order + list(missing_levels)

    fig, ax = plt.subplots(figsize=figsize)

    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=ycol,
        hue=color_col,
        style=style_col,
        palette=color_map,
        hue_order=hue_order,
        s=point_size,
        ax=ax,
    )

    if annotate:
        for idx, row in plot_df.iterrows():
            ax.text(row[xcol], row[ycol], str(idx), fontsize=8)

    if title is None:
        title = f"{xcol} vs {ycol}"
    ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.tick_params(length=3)

    if legend_outside and ax.legend_ is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend_.remove()
        fig.legend(
            handles,
            labels,
            bbox_to_anchor=legend_anchor,
            loc="upper left",
            frameon=False,
            borderaxespad=0.0,
        )
        plt.tight_layout(rect=[0, 0, main_width_ratio, 1])
    else:
        plt.tight_layout()

    if save is not None:
        fig.savefig(save, dpi=dpi, bbox_inches="tight")

    plt.show()
    return fig, ax


In [ ]:
res_composition = run_sample_composition_tripleplot(
    adata=T_all_tumor,
    sample_col="Sample_ID",
    subset_col="T_sub_level_3",
    meta_cols=("Period_merged", "Disease"),
    min_cells_per_sample=20,
    transform="clr",
    delta=None,
    center_features=True,
    scale=False,
    n_pcs=10,
    umap_n_neighbors=6,
    umap_min_dist=0.1,
    random_state=0,
    color_col="Disease",
    style_col="Period_merged",
    figsize=(18, 5),
    point_size=80,
    save=None,
)


In [ ]:
emb_umap = res_composition["emb_umap"]
sample_meta_plot = res_composition["sample_meta"].loc[emb_umap.index]

fig, ax = plot_sample_embedding(
    emb=emb_umap,
    sample_meta=sample_meta_plot,
    color_col="Disease",
    style_col="Period_merged",
    color_map=meta_color_map,
    figsize=(6, 4.5),
    point_size=80,
    title="",
    legend_outside=True,
    legend_anchor=(1.02, 1.0),
    main_width_ratio=0.76,
    save='figures/Fig5a_state_composition_umap.pdf',
    dpi=300,
)


# Fig5b

In [ ]:
def _fraction_p_to_stars(p: float) -> str:
    if np.isnan(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

def mixeddenom_mean_and_cooccurrence_heatmap(
    adata,
    sample_col: str = "Sample_ID",
    level1_col: str = "T_sub_level_1",
    level3_col: str = "T_sub_level_3",
    group_col: str = "Group",
    group_values: Optional[Union[str, Sequence[str]]] = None,

    drop_samples_min_totalT: int = 0,

    other_lineages: Sequence[str] = ("GD_T", "MAI_T", "Pro_T", "IM_T"),
    required_lineages: Sequence[str] = ("CD4_T", "CD8_T", "GD_T", "MAI_T", "Pro_T", "IM_T"),

    min_samples_per_pair: int = 5,
    fdr_method: str = "fdr_bh",

    cluster: bool = True,
    linkage_method: str = "average",

    cooccurrence_figsize=(10, 9),
    corr_vmin: float = -0.4,
    corr_vmax: float = 0.8,
    corr_cmap: str = "RdBu_r",
    cooccurrence_title: Optional[str] = None,
    x_label_rotation: float = 60,
    cbar_fraction: float = 0.035,
    cbar_pad: float = 0.02,
    draw_label_connectors: bool = True,
    connector_color: str = "black",
    connector_lw: float = 0.4,

    mean_figsize=(4, 8),
    mean_cmap: str = "Reds",
    mean_vmin: Optional[float] = None,
    mean_vmax: Optional[float] = None,
    mean_title: Optional[str] = None,
    annotate_mean: bool = True,
    mean_fmt: str = ".2f",

    save_cooccurrence_path: Optional[str] = None,
    save_mean_path: Optional[str] = None,
    dpi: int = 300,

    observed_groupby: bool = True,
):

    if not hasattr(adata, "obs"):
        raise TypeError("Input must be a scanpy AnnData object with .obs.")

    obs = adata.obs.copy()

    for c in [sample_col, level1_col, level3_col, group_col]:
        if c not in obs.columns:
            raise ValueError(f"adata.obs missing required column: {c}")

    if group_values is not None:
        if isinstance(group_values, str):
            group_values = [group_values]
        group_values = list(group_values)
        obs = obs.loc[obs[group_col].isin(group_values)].copy()

    for c in [sample_col, level1_col, level3_col, group_col]:
        if pd.api.types.is_categorical_dtype(obs[c]):
            obs[c] = obs[c].cat.remove_unused_categories()

    sample_group = (
        obs[[sample_col, group_col]]
        .drop_duplicates()
        .copy()
    )

    duplicated_samples = sample_group[sample_group.duplicated(sample_col, keep=False)]
    if not duplicated_samples.empty:
        raise ValueError(
            "Some samples have more than one group label. "
            "Please check sample_col and group_col."
        )

    sample_group = sample_group.set_index(sample_col)[group_col]

    totalT = (
        obs.groupby(sample_col, observed=observed_groupby)
        .size()
        .rename("totalT")
    )

    if drop_samples_min_totalT and drop_samples_min_totalT > 0:
        keep_samples = totalT[totalT > drop_samples_min_totalT].index
        obs = obs.loc[obs[sample_col].isin(keep_samples)].copy()

        totalT = (
            obs.groupby(sample_col, observed=observed_groupby)
            .size()
            .rename("totalT")
        )
        sample_group = sample_group.reindex(keep_samples).dropna()

    lin_tot = (
        obs.groupby([sample_col, level1_col], observed=observed_groupby)
        .size()
        .rename("n")
        .reset_index()
    )

    lin_tot_wide = lin_tot.pivot_table(
        index=sample_col,
        columns=level1_col,
        values="n",
        fill_value=0
    )

    for lin in required_lineages:
        if lin not in lin_tot_wide.columns:
            lin_tot_wide[lin] = 0

    lin_tot_wide["totalT"] = totalT.reindex(lin_tot_wide.index).fillna(0).astype(int)
    lin_tot_wide[group_col] = sample_group.reindex(lin_tot_wide.index)

    level3_str = obs[level3_col].astype(str)

    within_mask = (
        level3_str.str.startswith("CD4_")
        | level3_str.str.startswith("CD8_")
    )

    obs_within = obs.loc[
        within_mask,
        [sample_col, level3_col]
    ].copy()

    obs_within[level3_col] = obs_within[level3_col].astype(str)

    if obs_within.empty:
        feat_within = pd.DataFrame(
            index=[],
            columns=lin_tot_wide.index,
            dtype=float
        )
        feat_within.index.name = "feature"
        feat_within.columns.name = sample_col

    else:

        c3_counts = (
            obs_within
            .groupby([level3_col, sample_col], observed=observed_groupby)
            .size()
            .unstack(sample_col, fill_value=0)
        )

        c3_counts = (
            c3_counts
            .reindex(columns=lin_tot_wide.index, fill_value=0)
            .astype(float)
        )

        c3_counts.index = c3_counts.index.astype(str)
        c3_counts.index.name = "feature"
        c3_counts.columns.name = sample_col

        feat_within = pd.DataFrame(
            np.nan,
            index=c3_counts.index,
            columns=c3_counts.columns,
            dtype=float
        )

        feat_within.index.name = "feature"
        feat_within.columns.name = sample_col

        for feat in c3_counts.index:
            if feat.startswith("CD4_"):
                denom = lin_tot_wide["CD4_T"].reindex(c3_counts.columns).astype(float)
            elif feat.startswith("CD8_"):
                denom = lin_tot_wide["CD8_T"].reindex(c3_counts.columns).astype(float)
            else:
                continue

            counts = c3_counts.loc[feat].astype(float)

            feat_within.loc[feat] = np.where(
                denom > 0,
                counts / denom,
                np.nan
            )

    feat_other = {}

    for lin in other_lineages:
        num = lin_tot_wide[lin].astype(float)
        den = lin_tot_wide["totalT"].astype(float)

        frac = np.where(
            den > 0,
            num / den,
            np.nan
        )

        feat_other[lin] = pd.Series(frac, index=lin_tot_wide.index)

    feat_other_df = pd.DataFrame(feat_other).T
    feat_other_df.columns.name = sample_col

    sample_fraction = pd.concat([feat_within, feat_other_df], axis=0)

    sample_fraction = sample_fraction.loc[
        sample_fraction.notna().any(axis=1)
    ]

    var = sample_fraction.var(axis=1, skipna=True)
    sample_fraction = sample_fraction.loc[var > 0]

    features = sample_fraction.index.to_list()
    X = sample_fraction.to_numpy(dtype=float)
    n = len(features)

    corr = np.full((n, n), np.nan, float)
    pval = np.full((n, n), np.nan, float)

    for i in range(n):
        for j in range(i, n):
            xi = X[i, :]
            xj = X[j, :]
            m = np.isfinite(xi) & np.isfinite(xj)

            if m.sum() < max(3, min_samples_per_pair):
                continue

            if np.nanvar(xi[m]) == 0 or np.nanvar(xj[m]) == 0:
                continue

            r, p = spearmanr(xi[m], xj[m])
            corr[i, j] = corr[j, i] = r
            pval[i, j] = pval[j, i] = p

    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    ii, jj = np.where(mask)
    p_list = pval[ii, jj]
    valid = np.isfinite(p_list)

    p_adj = np.full((n, n), np.nan, float)

    if valid.any():
        _, p_adj_list, _, _ = multipletests(
            p_list[valid],
            method=fdr_method
        )

        p_adj[ii[valid], jj[valid]] = p_adj_list
        p_adj[jj[valid], ii[valid]] = p_adj_list

    np.fill_diagonal(p_adj, np.nan)

    stars = np.vectorize(_fraction_p_to_stars)(p_adj)
    np.fill_diagonal(stars, "")

    order = np.arange(n)

    if cluster and n > 1:
        corr_for_dist = np.nan_to_num(corr, nan=0.0)
        corr_for_dist = np.clip(corr_for_dist, -1.0, 1.0)

        dist = (1.0 - corr_for_dist) / 2.0
        np.fill_diagonal(dist, 0.0)

        condensed = squareform(dist, checks=False)
        Z = linkage(condensed, method=linkage_method)
        order = leaves_list(Z)

    features_ordered = [features[i] for i in order]

    corr_ord = corr[np.ix_(order, order)]
    p_adj_ord = p_adj[np.ix_(order, order)]
    stars_ord = stars[np.ix_(order, order)]

    corr_plot = corr_ord.copy()
    np.fill_diagonal(corr_plot, 1.0)

    sample_fraction_ordered = sample_fraction.loc[features_ordered]

    sample_fraction_long = (
        sample_fraction_ordered.T
        .reset_index()
        .rename(columns={sample_col: "Sample"})
        .melt(
            id_vars="Sample",
            var_name="Feature",
            value_name="Fraction"
        )
    )

    sample_fraction_long[group_col] = sample_fraction_long["Sample"].map(sample_group)

    group_mean = (
        sample_fraction_long
        .groupby(["Feature", group_col], observed=observed_groupby)["Fraction"]
        .mean()
        .unstack(group_col)
    )

    group_mean = group_mean.reindex(index=features_ordered)

    if group_values is not None:
        group_mean = group_mean.reindex(columns=group_values)

    if cooccurrence_title is None:
        if group_values is None:
            cooccurrence_title = "Co-occurrence correlation"
        else:
            cooccurrence_title = f"Co-occurrence correlation: {', '.join(map(str, group_values))}"

    if mean_title is None:
        mean_title = "Mean fraction by group"

    fig_corr, ax_corr = plt.subplots(figsize=cooccurrence_figsize)

    im_corr = ax_corr.imshow(
        corr_plot,
        vmin=corr_vmin,
        vmax=corr_vmax,
        cmap=corr_cmap,
        aspect="equal"
    )

    ax_corr.set_xticks(np.arange(n))
    ax_corr.set_yticks(np.arange(n))

    ax_corr.set_xticklabels(
        features_ordered,
        rotation=x_label_rotation,
        ha="right",
        fontsize=8
    )

    ax_corr.set_yticklabels(
        features_ordered,
        fontsize=8
    )

    for i in range(n):
        for j in range(n):
            s = stars_ord[i, j]
            if s:
                ax_corr.text(
                    j,
                    i,
                    s,
                    ha="center",
                    va="center",
                    fontsize=12,
                    fontweight="bold"
                )

    ax_corr.set_title(cooccurrence_title, fontsize=12, pad=10)

    cbar_corr = fig_corr.colorbar(
        im_corr,
        ax=ax_corr,
        fraction=cbar_fraction,
        pad=cbar_pad
    )

    cbar_corr.set_label(
        "Spearman correlation rho\n(feature fraction by sample)",
        rotation=90,
        fontsize=9
    )
    cbar_corr.ax.tick_params(labelsize=8, length=2)
    cbar_corr.outline.set_visible(False)

    for spine in ax_corr.spines.values():
        spine.set_visible(False)

    ax_corr.tick_params(length=0)

    if draw_label_connectors and n > 0:
        bottom_edge = n - 0.5
        left_edge = -0.5

        x_stub_len = 0.28
        for j in range(n):
            ax_corr.plot(
                [j, j],
                [bottom_edge, bottom_edge + x_stub_len],
                color=connector_color,
                lw=connector_lw,
                clip_on=False,
                solid_capstyle="butt",
                zorder=5
            )

        y_stub_len = 0.28
        for i in range(n):
            ax_corr.plot(
                [left_edge - y_stub_len, left_edge],
                [i, i],
                color=connector_color,
                lw=connector_lw,
                clip_on=False,
                solid_capstyle="butt",
                zorder=5
            )

    fig_corr.tight_layout()

    if save_cooccurrence_path:
        fig_corr.savefig(
            save_cooccurrence_path,
            dpi=dpi,
            bbox_inches="tight"
        )

    fig_mean, ax_mean = plt.subplots(figsize=mean_figsize)

    im_mean = ax_mean.imshow(
        group_mean.values,
        aspect="auto",
        cmap=mean_cmap,
        vmin=mean_vmin,
        vmax=mean_vmax
    )

    ax_mean.set_xticks(np.arange(group_mean.shape[1]))
    ax_mean.set_yticks(np.arange(group_mean.shape[0]))

    ax_mean.set_xticklabels(
        group_mean.columns.astype(str),
        fontsize=9
    )

    ax_mean.set_yticklabels(
        group_mean.index.astype(str),
        fontsize=8
    )

    ax_mean.set_xlabel(group_col, fontsize=10)
    ax_mean.set_ylabel("Cell type / compartment", fontsize=10)
    ax_mean.set_title(mean_title, fontsize=12, pad=10)

    if annotate_mean:
        for i in range(group_mean.shape[0]):
            for j in range(group_mean.shape[1]):
                val = group_mean.iloc[i, j]
                if pd.notna(val):
                    ax_mean.text(
                        j,
                        i,
                        format(val, mean_fmt),
                        ha="center",
                        va="center",
                        fontsize=7
                    )

    cbar_mean = fig_mean.colorbar(
        im_mean,
        ax=ax_mean,
        fraction=0.04,
        pad=0.03
    )

    cbar_mean.set_label("Mean fraction", fontsize=9)
    cbar_mean.ax.tick_params(labelsize=8)

    for spine in ax_mean.spines.values():
        spine.set_visible(False)

    ax_mean.tick_params(length=0)

    fig_mean.tight_layout()

    if save_mean_path:
        fig_mean.savefig(
            save_mean_path,
            dpi=dpi,
            bbox_inches="tight"
        )

    return {
        "sample_fraction": sample_fraction_ordered,
        "sample_fraction_with_group": sample_fraction_long,
        "group_mean": group_mean,
        "corr": pd.DataFrame(
            corr_ord,
            index=features_ordered,
            columns=features_ordered
        ),
        "p_adj": pd.DataFrame(
            p_adj_ord,
            index=features_ordered,
            columns=features_ordered
        ),
        "stars": pd.DataFrame(
            stars_ord,
            index=features_ordered,
            columns=features_ordered
        ),
        "order": features_ordered,
        "meta_counts": lin_tot_wide,
        "cooccurrence_figure": fig_corr,
        "cooccurrence_axis": ax_corr,
        "mean_fraction_figure": fig_mean,
        "mean_fraction_axis": ax_mean,
    }


In [ ]:
base_cmap = mpl.colormaps["RdBu_r"]

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_RdBu_r",
    base_cmap(np.linspace(0.10, 0.90, 256))
)

res = mixeddenom_mean_and_cooccurrence_heatmap(
    T_all_tumor,

    sample_col="Sample_ID",
    level1_col="T_sub_level_1",
    level3_col="T_sub_level_3",

    group_col="Period_merged",
    group_values=["Child", "Adult"],

    drop_samples_min_totalT=0,

    min_samples_per_pair=5,
    fdr_method="fdr_bh",

    cluster=True,
    linkage_method="average",

    cooccurrence_figsize=(10, 9),
    corr_vmin=-0.4,
    corr_vmax=0.8,
    corr_cmap=trunc_cmap,
    cooccurrence_title="",

    x_label_rotation=60,
    cbar_fraction=0.02,
    cbar_pad=0.02,

    draw_label_connectors=True,
    connector_color="black",
    connector_lw=0.8,

    mean_figsize=(4, 9),
    mean_cmap="Reds",
    mean_vmin=0,
    mean_vmax=None,
    mean_title="",

    annotate_mean=True,
    mean_fmt=".2f",

    save_cooccurrence_path='figures/Fig5b_fraction_cooccurrence.pdf',
    save_mean_path='figures/Fig5b_mean_fraction.pdf',

    dpi=300,
    observed_groupby=True
)


In [ ]:
cd4_exp = pd.read_csv('CD4_expansion_index_sample_cluster.csv')
cd8_exp = pd.read_csv('CD8_expansion_index_sample_cluster.csv')

cd4_exp = cd4_exp.rename(columns={"anno_CD4T": "Feature"})
cd8_exp = cd8_exp.rename(columns={"anno_CD8T": "Feature"})

exp_long = pd.concat([
    cd4_exp[["Sample_ID", "Period_merged", "Feature", "expansion_index"]],
    cd8_exp[["Sample_ID", "Period_merged", "Feature", "expansion_index"]]
], axis=0, ignore_index=True)

exp_matrix_raw = exp_long.pivot_table(
    index="Feature",
    columns="Sample_ID",
    values="expansion_index",
    aggfunc="first"
)

valid_samples = res["sample_fraction"].columns.intersection(exp_matrix_raw.columns)

expansion = pd.DataFrame(
    data=0.0,
    index=res["sample_fraction"].index.copy(),
    columns=valid_samples.copy()
)

common_features = expansion.index.intersection(exp_matrix_raw.index)
common_samples = expansion.columns.intersection(exp_matrix_raw.columns)

expansion.loc[common_features, common_samples] = (
    exp_matrix_raw.loc[common_features, common_samples]
    .fillna(0)
)

no_expansion_features = ["GD_T", "MAI_T", "IM_T", "Pro_T"]

for f in no_expansion_features:
    if f in expansion.index:
        expansion.loc[f, :] = np.nan

expansion


In [ ]:
def _p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

def plot_expansion_cooccurrence_from_matrix(
    expansion,
    sample_group,
    *,
    group_values=("Child", "Adult"),
    min_samples_per_pair=5,
    fdr_method="fdr_bh",

    corr_vmin=-0.4,
    corr_vmax=0.8,
    corr_cmap="RdBu_r",
    missing_color="lightgrey",

    cooccurrence_figsize=(10, 9),
    mean_figsize=(4, 9),

    x_label_rotation=60,
    cbar_fraction=0.02,
    cbar_pad=0.02,

    mean_cmap="Reds",
    mean_vmin=0,
    mean_vmax=None,
    annotate_mean=True,
    mean_fmt=".2f",

    save_cooccurrence_path=None,
    save_mean_path=None,
    dpi=300,
):

    expansion = expansion.copy()

    common_samples = expansion.columns.intersection(sample_group.index)
    expansion = expansion.loc[:, common_samples]

    if group_values is not None:
        group_values = list(group_values)
        keep_samples = [
            s for s in expansion.columns
            if sample_group.loc[s] in group_values
        ]
        expansion = expansion.loc[:, keep_samples]

    features = expansion.index.to_list()
    X = expansion.to_numpy(dtype=float)
    n = len(features)

    corr = np.full((n, n), np.nan, dtype=float)
    pval = np.full((n, n), np.nan, dtype=float)

    for i in range(n):
        for j in range(i, n):
            xi = X[i, :]
            xj = X[j, :]

            m = np.isfinite(xi) & np.isfinite(xj)

            if m.sum() < max(3, min_samples_per_pair):
                continue

            if np.nanstd(xi[m]) == 0 or np.nanstd(xj[m]) == 0:
                continue

            r, p = spearmanr(xi[m], xj[m])
            corr[i, j] = corr[j, i] = r
            pval[i, j] = pval[j, i] = p

    for i in range(n):
        xi = X[i, :]
        m = np.isfinite(xi)
        if m.sum() >= max(3, min_samples_per_pair) and np.nanstd(xi[m]) > 0:
            corr[i, i] = 1.0

    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    ii, jj = np.where(mask)

    p_list = pval[ii, jj]
    valid = np.isfinite(p_list)

    p_adj = np.full((n, n), np.nan, dtype=float)

    if valid.any():
        _, p_adj_list, _, _ = multipletests(
            p_list[valid],
            method=fdr_method
        )

        p_adj[ii[valid], jj[valid]] = p_adj_list
        p_adj[jj[valid], ii[valid]] = p_adj_list

    stars = np.vectorize(_p_to_stars)(p_adj)

    expansion_long = (
        expansion.T
        .reset_index()
        .rename(columns={"index": "Sample"})
        .melt(
            id_vars="Sample",
            var_name="Feature",
            value_name="expansion_index"
        )
    )

    expansion_long["Group"] = expansion_long["Sample"].map(sample_group)

    group_mean = (
        expansion_long
        .groupby(["Feature", "Group"], observed=True)["expansion_index"]
        .mean()
        .unstack("Group")
    )

    group_mean = group_mean.reindex(index=features)

    if group_values is not None:
        group_mean = group_mean.reindex(columns=group_values)

    corr_cmap_use = plt.get_cmap(corr_cmap).copy() if isinstance(corr_cmap, str) else corr_cmap.copy()
    corr_cmap_use.set_bad(color=missing_color)

    mean_cmap_use = plt.get_cmap(mean_cmap).copy() if isinstance(mean_cmap, str) else mean_cmap.copy()
    mean_cmap_use.set_bad(color=missing_color)

    corr_plot = np.ma.masked_invalid(corr)
    group_mean_plot = np.ma.masked_invalid(group_mean.values)

    fig_corr, ax_corr = plt.subplots(figsize=cooccurrence_figsize)

    im_corr = ax_corr.imshow(
        corr_plot,
        vmin=corr_vmin,
        vmax=corr_vmax,
        cmap=corr_cmap_use,
        aspect="equal"
    )

    ax_corr.set_xticks(np.arange(n))
    ax_corr.set_yticks(np.arange(n))

    ax_corr.set_xticklabels(
        features,
        rotation=x_label_rotation,
        ha="right",
        fontsize=8
    )

    ax_corr.set_yticklabels(
        features,
        fontsize=8
    )

    for i in range(n):
        for j in range(n):
            s = stars[i, j]
            if s:
                ax_corr.text(
                    j,
                    i,
                    s,
                    ha="center",
                    va="center",
                    fontsize=12,
                    fontweight="bold"
                )

    cbar_corr = fig_corr.colorbar(
        im_corr,
        ax=ax_corr,
        fraction=cbar_fraction,
        pad=cbar_pad
    )

    cbar_corr.set_label(
        "Spearman correlation rho\n(expansion index by sample)",
        rotation=90,
        fontsize=9
    )

    cbar_corr.ax.tick_params(labelsize=8, length=2)
    cbar_corr.outline.set_visible(False)

    for spine in ax_corr.spines.values():
        spine.set_visible(False)

    ax_corr.tick_params(length=0)
    fig_corr.tight_layout()

    if save_cooccurrence_path:
        fig_corr.savefig(
            save_cooccurrence_path,
            dpi=dpi,
            bbox_inches="tight"
        )

    fig_mean, ax_mean = plt.subplots(figsize=mean_figsize)

    im_mean = ax_mean.imshow(
        group_mean_plot,
        aspect="auto",
        cmap=mean_cmap_use,
        vmin=mean_vmin,
        vmax=mean_vmax
    )

    ax_mean.set_xticks(np.arange(group_mean.shape[1]))
    ax_mean.set_yticks(np.arange(group_mean.shape[0]))

    ax_mean.set_xticklabels(
        group_mean.columns.astype(str),
        fontsize=9
    )

    ax_mean.set_yticklabels(
        group_mean.index.astype(str),
        fontsize=8
    )

    ax_mean.set_xlabel("Group", fontsize=10)
    ax_mean.set_ylabel("Cell type / cluster", fontsize=10)

    if annotate_mean:
        for i in range(group_mean.shape[0]):
            for j in range(group_mean.shape[1]):
                val = group_mean.iloc[i, j]
                if pd.notna(val):
                    ax_mean.text(
                        j,
                        i,
                        format(val, mean_fmt),
                        ha="center",
                        va="center",
                        fontsize=7
                    )

    cbar_mean = fig_mean.colorbar(
        im_mean,
        ax=ax_mean,
        fraction=0.04,
        pad=0.03
    )

    cbar_mean.set_label("Mean expansion index", fontsize=9)
    cbar_mean.ax.tick_params(labelsize=8)

    for spine in ax_mean.spines.values():
        spine.set_visible(False)

    ax_mean.tick_params(length=0)
    fig_mean.tight_layout()

    if save_mean_path:
        fig_mean.savefig(
            save_mean_path,
            dpi=dpi,
            bbox_inches="tight"
        )

    return {
        "expansion": expansion,
        "expansion_long": expansion_long,
        "group_mean": group_mean,
        "corr": pd.DataFrame(corr, index=features, columns=features),
        "p_adj": pd.DataFrame(p_adj, index=features, columns=features),
        "stars": pd.DataFrame(stars, index=features, columns=features),
        "cooccurrence_figure": fig_corr,
        "cooccurrence_axis": ax_corr,
        "mean_expansion_figure": fig_mean,
        "mean_expansion_axis": ax_mean,
    }


In [ ]:
sample_group = (
    res["sample_fraction_with_group"][["Sample", "Period_merged"]]
    .drop_duplicates()
    .set_index("Sample")["Period_merged"]
)


In [ ]:
base_cmap = mpl.colormaps["RdBu_r"]

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_RdBu_r",
    base_cmap(np.linspace(0.10, 0.90, 256))
)

exp_res = plot_expansion_cooccurrence_from_matrix(
    expansion,
    sample_group,

    group_values=("Child", "Adult"),

    min_samples_per_pair=5,
    fdr_method="fdr_bh",

    corr_vmin=-0.4,
    corr_vmax=0.8,
    corr_cmap=trunc_cmap,
    missing_color="lightgrey",

    cooccurrence_figsize=(10, 9),
    mean_figsize=(4, 9),

    x_label_rotation=60,
    cbar_fraction=0.02,
    cbar_pad=0.02,

    mean_cmap="Reds",
    mean_vmin=0,
    mean_vmax=None,
    annotate_mean=True,
    mean_fmt=".2f",
    save_cooccurrence_path='figures/Fig5b_expansion_cooccurrence.pdf',
    save_mean_path='figures/Fig5b_mean_expansion.pdf',
    dpi=300
)


In [ ]:
order = res["corr"].index.tolist()
n = len(order)

corr_frac = res["corr"].reindex(index=order, columns=order)
p_frac = res["p_adj"].reindex(index=order, columns=order)

corr_exp = exp_res["corr"].reindex(index=order, columns=order)
p_exp = exp_res["p_adj"].reindex(index=order, columns=order)

combined_corr = pd.DataFrame(
    np.nan,
    index=order,
    columns=order
)

combined_padj = pd.DataFrame(
    np.nan,
    index=order,
    columns=order
)

upper_mask = np.triu(np.ones((n, n), dtype=bool), k=1)

lower_mask = np.tril(np.ones((n, n), dtype=bool), k=-1)

combined_corr.values[upper_mask] = corr_frac.values[upper_mask]
combined_padj.values[upper_mask] = p_frac.values[upper_mask]

combined_corr.values[lower_mask] = corr_exp.values[lower_mask]
combined_padj.values[lower_mask] = p_exp.values[lower_mask]

np.fill_diagonal(combined_corr.values, np.nan)
np.fill_diagonal(combined_padj.values, np.nan)

combined_stars = combined_padj.applymap(_p_to_stars)

base_cmap = mpl.colormaps["RdBu_r"]

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_RdBu_r",
    base_cmap(np.linspace(0.10, 0.90, 256))
)

cmap_use = trunc_cmap.copy()
cmap_use.set_bad(color="#e6e6e6")

plot_mat = np.ma.masked_invalid(combined_corr.values)

figsize = (10, 9)

corr_vmin = -0.4
corr_vmax = 0.8

x_label_rotation = 60

cbar_fraction = 0.02
cbar_pad = 0.02

draw_label_connectors = True
connector_color = "black"
connector_lw = 0.8
x_stub_len = 0.28
y_stub_len = 0.28

fig, ax = plt.subplots(figsize=figsize)

im = ax.imshow(
    plot_mat,
    cmap=cmap_use,
    vmin=corr_vmin,
    vmax=corr_vmax,
    aspect="equal"
)

ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))

ax.set_xticklabels(
    order,
    rotation=x_label_rotation,
    ha="right",
    fontsize=8
)

ax.set_yticklabels(
    order,
    fontsize=8
)

for i in range(n):
    for j in range(n):
        s = combined_stars.iloc[i, j]
        if s != "":
            ax.text(
                j,
                i,
                s,
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color="black"
            )

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(length=0)

if draw_label_connectors and n > 0:
    bottom_edge = n - 0.5
    left_edge = -0.5

    for j in range(n):
        ax.plot(
            [j, j],
            [bottom_edge, bottom_edge + x_stub_len],
            color=connector_color,
            lw=connector_lw,
            clip_on=False,
            solid_capstyle="butt",
            zorder=5
        )

    for i in range(n):
        ax.plot(
            [left_edge - y_stub_len, left_edge],
            [i, i],
            color=connector_color,
            lw=connector_lw,
            clip_on=False,
            solid_capstyle="butt",
            zorder=5
        )

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=cbar_fraction,
    pad=cbar_pad
)

cbar.set_label(
    "Spearman correlation rho",
    fontsize=9
)

cbar.ax.tick_params(
    labelsize=8,
    length=2
)

cbar.outline.set_visible(False)

plt.tight_layout()

plt.savefig(
    'figures/Fig5b_combined_cooccurrence.pdf',
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# Fig5c

In [ ]:
def get_sample_metadata(
    adata,
    sample_col="Sample_ID",
    meta_cols=("Period_merged", "Disease")
):
    obs_cols = [sample_col] + list(meta_cols)
    meta = adata.obs[obs_cols].drop_duplicates().copy()

    dup_check = meta.groupby(sample_col).size()
    if (dup_check > 1).any():
        bad = dup_check[dup_check > 1].index.tolist()[:10]
        raise ValueError(f"Some {sample_col} map to multiple metadata rows, e.g. {bad}")

    return meta.set_index(sample_col)


In [ ]:
def generate_pairs_to_test(
    res,
    p_cutoff=None,
    min_abs_rho=None,
    include_positive=True,
    include_negative=True,
):
    corr = res["corr"]
    p_adj = res["p_adj"]

    out = []
    feats = corr.index.tolist()

    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            f1, f2 = feats[i], feats[j]
            rho = corr.loc[f1, f2]
            p = p_adj.loc[f1, f2]

            if pd.isna(rho):
                continue

            if p_cutoff is not None:
                if pd.isna(p) or p > p_cutoff:
                    continue

            if min_abs_rho is not None and abs(rho) < min_abs_rho:
                continue

            if rho > 0 and not include_positive:
                continue
            if rho < 0 and not include_negative:
                continue

            out.append((f1, f2, rho, p))

    pair_df = pd.DataFrame(out, columns=["feature1", "feature2", "rho", "p_adj"])

    if len(pair_df) == 0:
        return pair_df, []

    pair_df = pair_df.sort_values("rho", ascending=False).reset_index(drop=True)
    pairs_to_test = list(pair_df[["feature1", "feature2"]].itertuples(index=False, name=None))

    return pair_df, pairs_to_test


In [ ]:
def _permanova_r2_two_groups(X, group_labels, n_perm=999, random_state=0):
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)
    g = pd.Series(group_labels).astype(str).values

    grand_centroid = X.mean(axis=0)
    ss_total = ((X - grand_centroid) ** 2).sum()

    ss_between = 0.0
    for lv in np.unique(g):
        Xi = X[g == lv]
        ci = Xi.mean(axis=0)
        ss_between += len(Xi) * ((ci - grand_centroid) ** 2).sum()

    ss_within = ss_total - ss_between
    df_between = len(np.unique(g)) - 1
    df_within = len(X) - len(np.unique(g))

    ms_between = ss_between / df_between
    ms_within = ss_within / df_within if df_within > 0 else np.nan
    F_obs = ms_between / ms_within if ms_within > 0 else np.nan
    R2 = ss_between / ss_total if ss_total > 0 else np.nan

    perm_F = []
    for _ in range(n_perm):
        g_perm = rng.permutation(g)
        ss_between_perm = 0.0
        for lv in np.unique(g_perm):
            Xi = X[g_perm == lv]
            ci = Xi.mean(axis=0)
            ss_between_perm += len(Xi) * ((ci - grand_centroid) ** 2).sum()
        ss_within_perm = ss_total - ss_between_perm
        ms_between_perm = ss_between_perm / df_between
        ms_within_perm = ss_within_perm / df_within if df_within > 0 else np.nan
        F_perm = ms_between_perm / ms_within_perm if ms_within_perm > 0 else np.nan
        perm_F.append(F_perm)

    perm_F = np.array(perm_F, dtype=float)
    pval = (np.sum(perm_F >= F_obs) + 1) / (np.sum(np.isfinite(perm_F)) + 1)

    return {"F": F_obs, "R2": R2, "pval": pval}

def evaluate_period_effect_on_pairs(
    res,
    sample_meta,
    period_col="Period_merged",
    pairs=None,
    positive_label="Adult",
    n_perm=999,
    random_state=0,
):

    feat_mat = res["sample_fraction"]

    if pairs is None:
        raise ValueError("Please provide pairs=[(feature1, feature2), ...]")

    out = []

    for feature1, feature2 in pairs:
        if feature1 not in feat_mat.index or feature2 not in feat_mat.index:
            continue

        df = pd.DataFrame({
            feature1: feat_mat.loc[feature1],
            feature2: feat_mat.loc[feature2],
        }).T.T

        df.index = feat_mat.columns
        df = df.join(sample_meta[[period_col]], how="left").dropna()

        if df[period_col].nunique() < 2:
            continue

        x = df[feature1].values
        y = df[feature2].values
        g = df[period_col].astype(str).values

        rho_raw, p_raw = spearmanr(x, y)

        g01 = (df[period_col].astype(str) == positive_label).astype(int).values.reshape(-1, 1)
        lm_x = LinearRegression().fit(g01, x)
        lm_y = LinearRegression().fit(g01, y)
        x_resid = x - lm_x.predict(g01)
        y_resid = y - lm_y.predict(g01)
        rho_resid, p_resid = spearmanr(x_resid, y_resid)

        if np.isfinite(rho_raw) and abs(rho_raw) > 1e-12:
            corr_explained = 1 - (abs(rho_resid) / abs(rho_raw))
        else:
            corr_explained = np.nan

        X2 = df[[feature1, feature2]].values
        perm_res = _permanova_r2_two_groups(
            X2, g, n_perm=n_perm, random_state=random_state
        )

        y_bin = (df[period_col].astype(str) == positive_label).astype(int).values
        clf = LogisticRegression(max_iter=1000)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
        prob = cross_val_predict(clf, X2, y_bin, cv=cv, method="predict_proba")[:, 1]
        auc = roc_auc_score(y_bin, prob)

        out.append({
            "feature1": feature1,
            "feature2": feature2,
            "n_samples": len(df),
            "rho_raw": rho_raw,
            "p_raw": p_raw,
            "rho_resid": rho_resid,
            "p_resid": p_resid,
            "corr_explained_by_period": corr_explained,
            "pair_PERMANOVA_R2": perm_res["R2"],
            "pair_PERMANOVA_p": perm_res["pval"],
            "pair_auc": auc,
        })

    return pd.DataFrame(out).sort_values(
        ["corr_explained_by_period", "pair_PERMANOVA_R2"],
        ascending=False
    )


In [ ]:
def _add_confidence_ellipse(
    ax,
    x,
    y,
    edgecolor="black",
    facecolor="none",
    alpha=0.12,
    lw=1.5,
    conf=0.80,
    zorder=1
):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 3:
        return

    cov = np.cov(x, y)

    if np.linalg.matrix_rank(cov) < 2:
        return

    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    scale = np.sqrt(chi2.ppf(conf, df=2))

    width, height = 2 * scale * np.sqrt(vals)
    mean_x, mean_y = np.mean(x), np.mean(y)

    ell = Ellipse(
        (mean_x, mean_y),
        width=width,
        height=height,
        angle=angle,
        edgecolor=edgecolor,
        facecolor=facecolor,
        alpha=alpha if facecolor != "none" else 1.0,
        lw=lw,
        zorder=zorder,
    )

    ax.add_patch(ell)

def _get_pair_eval_stats(pair_eval, feature1, feature2):
    out = {
        "rho_raw": np.nan,
        "corr_explained_by_period": np.nan,
        "pair_auc": np.nan,
        "found": False,
    }

    if pair_eval is None:
        return out

    if not isinstance(pair_eval, pd.DataFrame):
        raise TypeError("pair_eval must be a pandas DataFrame.")

    required_cols = {
        "feature1",
        "feature2",
        "rho_raw",
        "corr_explained_by_period",
        "pair_auc",
    }

    missing_cols = required_cols - set(pair_eval.columns)

    if len(missing_cols) > 0:
        raise ValueError(f"pair_eval missing required columns: {missing_cols}")

    f1 = str(feature1)
    f2 = str(feature2)

    pe = pair_eval.copy()
    pe["feature1"] = pe["feature1"].astype(str)
    pe["feature2"] = pe["feature2"].astype(str)

    hit = pe[
        (
            (pe["feature1"] == f1)
            & (pe["feature2"] == f2)
        )
        |
        (
            (pe["feature1"] == f2)
            & (pe["feature2"] == f1)
        )
    ]

    if len(hit) == 0:
        return out

    row = hit.iloc[0]

    out["rho_raw"] = row["rho_raw"]
    out["corr_explained_by_period"] = row["corr_explained_by_period"]
    out["pair_auc"] = row["pair_auc"]
    out["found"] = True

    return out

def plot_cooccurrence_pair(
    res,
    feature1,
    feature2,
    sample_meta=None,
    pair_eval=None,
    color_col=None,
    style_col=None,
    color_map=None,
    meta_color_map=None,
    period_col="Period_merged",

    figsize=(9.2, 5.2),
    fixed_layout=True,
    ax_rect=(0.12, 0.16, 0.58, 0.72),
    legend_rect=(0.74, 0.16, 0.24, 0.72),

    title=None,
    legend_outside=True,
    point_size=70,
    alpha=0.95,
    xlim=None,
    ylim=None,
    savepath=None,
    dpi=300,

    show_overall_reg=True,
    reg_lw=1.8,
    reg_color="black",
    reg_linestyle="--",

    show_group_ellipse=True,
    ellipse_conf=0.80,
    ellipse_alpha=0.10,
    ellipse_lw=1.4,

    show_stats_text=True,
    stats_text_pos=(0.97, 0.96),
    stats_fontsize=10,
    stats_label_style="compact",

    point_edgecolor="none",
    point_linewidth=0,

    axis_lw=1.35,
    tick_width=1.2,
    tick_length=4,
    tick_labelsize=9,
    axis_labelsize=10,
    title_fontsize=11,

    show=True,
    save_bbox_inches=None,
):

    if "sample_fraction" not in res:
        raise KeyError("res must contain res['sample_fraction'].")

    feat_mat = res["sample_fraction"]

    if feature1 not in feat_mat.index:
        raise ValueError(f"{feature1!r} not found in res['sample_fraction'].index")

    if feature2 not in feat_mat.index:
        raise ValueError(f"{feature2!r} not found in res['sample_fraction'].index")

    df = pd.DataFrame({
        feature1: feat_mat.loc[feature1],
        feature2: feat_mat.loc[feature2],
    }).copy()

    if sample_meta is not None:
        if not isinstance(sample_meta, pd.DataFrame):
            raise TypeError("sample_meta must be a pandas DataFrame.")

        df = df.join(sample_meta, how="left")

    df = df.dropna(subset=[feature1, feature2]).copy()

    stats = _get_pair_eval_stats(pair_eval, feature1, feature2)

    rho_raw = stats["rho_raw"]
    corr_explained = stats["corr_explained_by_period"]
    pair_auc = stats["pair_auc"]

    if not stats["found"]:
        if (
            "corr" in res
            and feature1 in res["corr"].index
            and feature2 in res["corr"].columns
        ):
            rho_raw = res["corr"].loc[feature1, feature2]

        elif (
            len(df) >= 3
            and df[feature1].nunique() > 1
            and df[feature2].nunique() > 1
        ):
            rho_raw, _ = spearmanr(df[feature1], df[feature2])

    hue_order = None

    if color_col is not None and color_col in df.columns:
        df[color_col] = df[color_col].astype(str)

        if color_map is not None:
            color_map = {str(k): v for k, v in color_map.items()}

            present_levels = pd.unique(df[color_col].dropna())
            hue_order = [k for k in color_map.keys() if k in present_levels]

            missing_levels = [x for x in present_levels if x not in color_map]
            hue_order = hue_order + list(missing_levels)

    if style_col is not None and style_col in df.columns:
        df[style_col] = df[style_col].astype(str)

    if fixed_layout:
        fig = plt.figure(figsize=figsize, constrained_layout=False)
        ax = fig.add_axes(ax_rect)

        if legend_outside:
            legend_ax = fig.add_axes(legend_rect)
            legend_ax.axis("off")
        else:
            legend_ax = None

    else:
        fig, ax = plt.subplots(figsize=figsize)
        legend_ax = None

    sns.scatterplot(
        data=df,
        x=feature1,
        y=feature2,
        hue=color_col,
        style=style_col,
        palette=color_map,
        hue_order=hue_order,
        s=point_size,
        alpha=alpha,
        ax=ax,
        zorder=3,
        edgecolor=point_edgecolor,
        linewidth=point_linewidth,
    )

    for coll in ax.collections:
        try:
            coll.set_linewidths(0)
            coll.set_edgecolors("none")
        except Exception:
            pass

    if show_overall_reg and len(df) >= 3:
        sns.regplot(
            data=df,
            x=feature1,
            y=feature2,
            scatter=False,
            ci=None,
            truncate=True,
            line_kws={
                "linestyle": reg_linestyle,
                "linewidth": reg_lw,
                "color": reg_color,
            },
            ax=ax,
        )

    if (
        show_group_ellipse
        and period_col in df.columns
        and df[period_col].dropna().nunique() >= 2
    ):
        groups = list(df[period_col].dropna().astype(str).unique())

        if meta_color_map is not None:
            meta_color_map = {str(k): v for k, v in meta_color_map.items()}

            pal = {}
            missing_groups = []

            for g in groups:
                if g in meta_color_map:
                    pal[g] = meta_color_map[g]
                else:
                    missing_groups.append(g)

            if len(missing_groups) > 0:
                extra = sns.color_palette(
                    "Dark2",
                    n_colors=len(missing_groups)
                ).as_hex()

                for g, c in zip(missing_groups, extra):
                    pal[g] = c
        else:
            pal = dict(
                zip(
                    groups,
                    sns.color_palette("Dark2", n_colors=len(groups)).as_hex()
                )
            )

        for g in groups:
            dfg = df[df[period_col].astype(str) == g].copy()

            if len(dfg) >= 3:
                _add_confidence_ellipse(
                    ax,
                    dfg[feature1].values,
                    dfg[feature2].values,
                    edgecolor=pal[g],
                    facecolor=pal[g],
                    alpha=ellipse_alpha,
                    lw=ellipse_lw,
                    conf=ellipse_conf,
                    zorder=1,
                )

    if title is None:
        title = f"{feature1} vs {feature2}"

    ax.set_title(title, fontsize=title_fontsize, pad=8)

    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

    if xlim is not None:
        ax.set_xlim(xlim)

    if ylim is not None:
        ax.set_ylim(ylim)

    if show_stats_text:
        rho_txt = f"{rho_raw:.2f}" if np.isfinite(rho_raw) else "NA"

        if np.isfinite(corr_explained):
            age_explained_txt = f"{corr_explained * 100:.0f}%"
        else:
            age_explained_txt = "NA"

        auc_txt = f"{pair_auc:.2f}" if np.isfinite(pair_auc) else "NA"

        if stats_label_style == "compact":
            stat_txt = (
                f"rho_raw = {rho_txt}\n"
                f"Age-driven = {age_explained_txt}\n"
                f"AUC = {auc_txt}"
            )
        else:
            stat_txt = (
                f"rho_raw = {rho_txt}\n"
                f"Age explained = {age_explained_txt}\n"
                f"AUC = {auc_txt}"
            )

        ax.text(
            stats_text_pos[0],
            stats_text_pos[1],
            stat_txt,
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=stats_fontsize,
            color="black",
            zorder=10,
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)

    ax.spines["left"].set_linewidth(axis_lw)
    ax.spines["bottom"].set_linewidth(axis_lw)

    ax.tick_params(
        axis="both",
        length=tick_length,
        width=tick_width,
        labelsize=tick_labelsize,
    )

    ax.xaxis.label.set_size(axis_labelsize)
    ax.yaxis.label.set_size(axis_labelsize)

    if ax.legend_ is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend_.remove()

        if legend_outside:
            if fixed_layout and legend_ax is not None:
                legend_ax.legend(
                    handles,
                    labels,
                    loc="upper left",
                    frameon=False,
                    borderaxespad=0.0,
                    fontsize=9,
                    handletextpad=0.4,
                    labelspacing=0.5,
                )
            else:
                fig.legend(
                    handles,
                    labels,
                    bbox_to_anchor=(0.74, 0.88),
                    loc="upper left",
                    frameon=False,
                    borderaxespad=0.0,
                    fontsize=9,
                )
        else:
            ax.legend(
                handles,
                labels,
                loc="best",
                frameon=False,
                fontsize=9,
            )

    if savepath is not None:
        fig.savefig(
            savepath,
            dpi=dpi,
            bbox_inches=save_bbox_inches,
        )

    if show:
        plt.show()

    return fig, ax, df


In [ ]:
cooccurrence_sample_meta = get_sample_metadata(
    T_all_tumor,
    sample_col="Sample_ID",
    meta_cols=("Period_merged", "Disease")
)


In [ ]:
pair_df, pairs_to_test = generate_pairs_to_test(
    res,
    p_cutoff=0.001,
    include_positive=True,
    include_negative=True,
)

pair_eval = evaluate_period_effect_on_pairs(
    res=res,
    sample_meta=cooccurrence_sample_meta,
    period_col="Period_merged",
    pairs=pairs_to_test,
    positive_label="Adult",
    n_perm=999,
    random_state=0,
)

pair_eval
pair_eval.to_csv('results/pair_eval.csv', index=False)


In [ ]:
pairs_to_plot = [
    ("GD_T", "CD8_TEX_c12"),
    ("GD_T", "CD4_TFH_c8"),
    ("CD8_t-TEM_c8", "CD8_TEM_c5"),
    ("CD4_TFH_c8", "CD8_TEX_c12"),
]

for f1, f2 in pairs_to_plot:
    plot_cooccurrence_pair(
        res=res,
        feature1=f1,
        feature2=f2,

        sample_meta=cooccurrence_sample_meta,
        pair_eval=pair_eval,

        color_col="Disease",
        style_col="Period_merged",
        color_map=meta_color_map,
        meta_color_map=meta_color_map,
        period_col="Period_merged",

        figsize=(9, 6),
        fixed_layout=True,
        ax_rect=(0.12, 0.15, 0.58, 0.72),
        legend_rect=(0.74, 0.15, 0.24, 0.72),

        show_stats_text=True,
        stats_label_style="compact",
        stats_text_pos=(0.96, 0.96),
        stats_fontsize=10,

        axis_lw=1.35,
        tick_width=1.2,
        tick_length=4,

        dpi=300,
        save_bbox_inches=None,

        show=True,
    )


# FigS5g

In [ ]:
T_cnmf = sc.read_h5ad( 'T.h5ad')

T_cnmf_counts = T_cnmf.raw.to_adata()
T_cnmf_counts.obs = T_cnmf.obs.copy()

hvg_genes = T_cnmf.var.index[T_cnmf.var["highly_variable_features"].astype(bool)].tolist()

hvg_genes = [g for g in hvg_genes if g in T_cnmf_counts.var_names]

T_cnmf_hvg = T_cnmf_counts[:, hvg_genes].copy()

assert "batch" in T_cnmf_hvg.obs.columns
T_cnmf_hvg.write_h5ad('results/cnmf/T_hvg.h5ad', compression='gzip')


In [ ]:
T_cnmf_hvg = ad.read_h5ad('results/cnmf/T_hvg.h5ad')
out_dir = 'results/cnmf/k_scan'
os.makedirs(out_dir, exist_ok=True)
run_base = os.path.join(out_dir, "T_hc")
p = Preprocess(random_seed=14)
adata_c, adata_tp10k, hvgs = p.preprocess_for_cnmf(
    T_cnmf_hvg,
    harmony_vars="batch",
    n_top_rna_genes=2000,
    max_scaled_thresh=None,
    quantile_thresh=0.9999,
    makeplots=True,
    save_output_base=run_base
)
print("DONE. Outputs written with prefix:", run_base)


In [ ]:
out_dir = 'results/cnmf/k_scan'
name = "T_hc"
os.makedirs(out_dir, exist_ok=True)

cnmf_obj = cNMF(output_dir=out_dir, name=name)

Ks = np.arange(10, 41)

run_base  = os.path.join(out_dir, "T_hc")
counts_fn = run_base + ".Corrected.HVG.Varnorm.h5ad"
tpm_fn    = run_base + ".TP10K.h5ad"
genes_fn  = run_base + ".Corrected.HVGs.txt"

cnmf_obj.prepare(
    counts_fn=counts_fn,
    tpm_fn=tpm_fn,
    genes_file=genes_fn,
    components=Ks,
    n_iter=20,
    seed=14,
    num_highvar_genes=2000
)

print("prepare done:", cnmf_obj.output_dir)


In [ ]:
worker_i = 0
total_workers = 1

out_dir = 'results/cnmf/k_scan'
name = "T_hc"

cnmf_obj = cNMF(output_dir=out_dir, name=name)
cnmf_obj.factorize(worker_i=worker_i, total_workers=total_workers)

print(f"factorize done: worker {worker_i}/{total_workers}")


In [ ]:
out_dir = 'results/cnmf/k_scan'
name = "T_hc"

cnmf_obj = cNMF(output_dir=out_dir, name=name)

cnmf_obj.combine()
print("combine done:", cnmf_obj.output_dir)

cnmf_obj.k_selection_plot(close_fig=False)

fig_dir = "figures"
os.makedirs(fig_dir, exist_ok=True)

png_path = os.path.join(fig_dir, 'FigS5g_cNMF_k_selection.png')
pdf_path = os.path.join(fig_dir, 'FigS5g_cNMF_k_selection.pdf')

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.close("all")

print("Saved:", png_path)
print("Saved:", pdf_path)


# FigS5h

In [ ]:
T_cnmf_hvg = ad.read_h5ad('results/cnmf/T_hvg.h5ad')
out_dir = 'results/cnmf/k27'
os.makedirs(out_dir, exist_ok=True)
run_base = os.path.join(out_dir, "T_hc")
p = Preprocess(random_seed=14)
adata_c, adata_tp10k, hvgs = p.preprocess_for_cnmf(
    T_cnmf_hvg,
    harmony_vars="batch",
    n_top_rna_genes=2000,
    max_scaled_thresh=None,
    quantile_thresh=0.9999,
    makeplots=True,
    save_output_base=run_base
)
print("DONE. Outputs written with prefix:", run_base)


In [ ]:
out_dir = 'results/cnmf/k27'
name = "T_hc"
os.makedirs(out_dir, exist_ok=True)

cnmf_obj = cNMF(output_dir=out_dir, name=name)

Ks=27
run_base  = os.path.join(out_dir, "T_hc")
counts_fn = run_base + ".Corrected.HVG.Varnorm.h5ad"
tpm_fn    = run_base + ".TP10K.h5ad"
genes_fn  = run_base + ".Corrected.HVGs.txt"

cnmf_obj.prepare(
    counts_fn=counts_fn,
    tpm_fn=tpm_fn,
    genes_file=genes_fn,
    components=Ks,
    n_iter=200,
    seed=14,
    num_highvar_genes=2000
)

print("prepare done:", cnmf_obj.output_dir)


In [ ]:
worker_i = 0
total_workers = 1

out_dir = 'results/cnmf/k27'
name = "T_hc"

cnmf_obj = cNMF(output_dir=out_dir, name=name)
cnmf_obj.factorize(worker_i=worker_i, total_workers=total_workers)

print(f"factorize done: worker {worker_i}/{total_workers}")


In [ ]:
out_dir = 'results/cnmf/k27'
name = "T_hc"

cnmf_obj = cNMF(output_dir=out_dir, name=name)

cnmf_obj.combine()
print("combine done:", cnmf_obj.output_dir)

cnmf_obj.k_selection_plot(close_fig=False)

fig_dir = "figures"
os.makedirs(fig_dir, exist_ok=True)

png_path = os.path.join(fig_dir, 'FigS5h_cNMF_k27_selection.png')
pdf_path = os.path.join(fig_dir, 'FigS5h_cNMF_k27_selection.pdf')

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.close("all")

print("Saved:", png_path)
print("Saved:", pdf_path)


In [ ]:
out_dir = 'results/cnmf/k27'
name = "T_hc"

cnmf_obj = cNMF(output_dir=out_dir, name=name)
Ks_to_run = [27]
density_threshold = 0.1

for k in Ks_to_run:
    print(f"\n=== Running consensus for k={k} (dt={density_threshold}) ===")
    cnmf_obj.consensus(
        k=k,
        density_threshold=density_threshold,
        show_clustering=True,
        close_clustergram_fig=False
    )

    plt.close("all")
    print(f"Done consensus k={k}")

print("\nAll consensus runs finished.")
print("Output dir:", cnmf_obj.output_dir)


In [ ]:
selected_K=27
out_dir = 'results/cnmf/k27'
name = "T_hc"
k = selected_K
dt = 0.1

cnmf_obj = cNMF(output_dir=out_dir, name=name)
usage, spectra_scores, spectra_tpm, top_genes = cnmf_obj.load_results(K=k, density_threshold=dt)


In [ ]:
usage = usage.loc[T_cnmf.obs_names]

for p in range(1, k+1):
    T_cnmf.obs[f"Program_{p}"] = usage[p].values if p in usage.columns else usage.iloc[:, p-1].values


In [ ]:
def compute_subset_program_similarity(
    adata,
    subset_col="T_sub_level_3",
    program_prefix="Program_",
    n_programs=27,
    min_cells=20,
    metric="correlation",
    linkage_method="average",
    zscore_programs=False,
    return_dendrogram=True,
):

    if subset_col not in adata.obs.columns:
        raise ValueError(f"{subset_col} not found in adata.obs")

    program_cols = [f"{program_prefix}{i}" for i in range(1, n_programs + 1)]
    missing = [c for c in program_cols if c not in adata.obs.columns]
    if len(missing) > 0:
        raise ValueError(f"Missing program columns: {missing}")

    df = adata.obs[[subset_col] + program_cols].copy()
    df = df[df[subset_col].notna()].copy()

    subset_sizes = df[subset_col].value_counts()
    keep_subsets = subset_sizes[subset_sizes >= min_cells].index.tolist()
    if len(keep_subsets) < 2:
        raise ValueError(f"Need at least 2 subsets with >= {min_cells} cells")

    df = df[df[subset_col].isin(keep_subsets)].copy()
    subset_sizes = df[subset_col].value_counts().sort_index()

    mean_usage = df.groupby(subset_col)[program_cols].mean()
    mean_usage = mean_usage.loc[sorted(mean_usage.index)]

    if zscore_programs:
        mean_usage = mean_usage.apply(
            lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else 0,
            axis=0
        )

    dist_vec = pdist(mean_usage.values, metric=metric)
    dist_mat = squareform(dist_vec)
    distance_matrix = pd.DataFrame(
        dist_mat,
        index=mean_usage.index,
        columns=mean_usage.index
    )

    Z = linkage(dist_vec, method=linkage_method)

    subset_order = mean_usage.index.tolist()
    dendro = None
    if return_dendrogram:
        dendro = dendrogram(Z, labels=mean_usage.index.tolist(), no_plot=True)
        subset_order = dendro["ivl"]

    return {
        "mean_usage": mean_usage,
        "distance_matrix": distance_matrix,
        "linkage": Z,
        "subset_sizes": subset_sizes,
        "subset_order": subset_order,
        "dendrogram": dendro,
        "program_cols": program_cols,
    }


In [ ]:
def plot_program_usage_heatmap(
    res,
    figsize=(10, 8),
    zscore_rows=False,
    cmap="RdBu_r",
    center=None,
    title="Mean cNMF program usage across T cell subsets",
    xlabel="Programs",
    ylabel="T cell subsets",
    annot=False,
    fmt=".2f",
    linewidths=0,
    linecolor="white",
    cbar=True,
    xtick_rotation=90,
    ytick_rotation=0
):

    order = res["subset_order"]
    mat = res["mean_usage"].loc[order].copy()

    if zscore_rows:
        row_std = mat.std(axis=1).replace(0, np.nan)
        mat = mat.sub(mat.mean(axis=1), axis=0).div(row_std, axis=0).fillna(0)

    if center is None and zscore_rows:
        center = 0

    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        mat,
        cmap=cmap,
        center=center,
        annot=annot,
        fmt=fmt,
        linewidths=linewidths,
        linecolor=linecolor,
        cbar=cbar,
        ax=ax
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=xtick_rotation, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=ytick_rotation)

    plt.tight_layout()

    return fig, ax


In [ ]:
def plot_subset_dendrogram(
    res,
    figsize=(12, 5),
    leaf_rotation=90,
    leaf_fontsize=10,
    color_threshold=None,
    title="T cell subset clustering based on cNMF programs",
    ylabel="Distance",
    save=None,
    show=True,
    palette=None,
    above_threshold_color="gray",
):
    fig, ax = plt.subplots(figsize=figsize)

    if palette is not None:
        if isinstance(palette, str):
            colors = list(sns.color_palette(palette, 12).as_hex())
        else:
            colors = list(palette)
        set_link_color_palette(colors)

    if color_threshold is None:
        color_threshold = 0.7 * max(res["linkage"][:, 2])

    dendro = dendrogram(
        res["linkage"],
        labels=res["mean_usage"].index.tolist(),
        leaf_rotation=leaf_rotation,
        leaf_font_size=leaf_fontsize,
        color_threshold=color_threshold,
        above_threshold_color=above_threshold_color,
        ax=ax,
    )

    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.tight_layout()

    if save is not None:
        fig.savefig(save, bbox_inches="tight")

    if show:
        plt.show()

    set_link_color_palette(None)

    return fig, ax, dendro


In [ ]:
state_program_res = compute_subset_program_similarity(
    T_cnmf,
    subset_col="T_sub_level_3",
    program_prefix="Program_",
    n_programs=27,
    min_cells=20,
    metric="correlation",
    linkage_method="average",
    zscore_programs=True,
)


In [ ]:
fig, ax = plot_program_usage_heatmap(
    state_program_res,
    figsize=(10, 8),
    zscore_rows=True
)

fig.savefig(
    'figures/FigS5h_state_program_usage.pdf',
    format="pdf",
    bbox_inches="tight"
)
plt.show()


In [ ]:
fig, ax, dendro = plot_subset_dendrogram(
    state_program_res,
    figsize=(12, 5),
    palette="Set1",
    color_threshold=0.7,
    save='figures/FigS5h_state_program_dendrogram.pdf'
)


# Fig5h

In [ ]:
T_cnmf_tumor = T_cnmf[T_cnmf.obs["Sample_Type"].isin(["T"])].copy()


In [ ]:
def plot_program_usage_heatmap_with_annotations(
    res,
    adata,
    subset_col="Sample_ID",
    anno_cols=("Period_merged", "Disease"),
    color_map=None,
    figsize=(10, 8),
    zscore_rows=True,
    cmap="RdBu_r",
    center=0,
    heatmap_cbar_label="Program usage (row z-score)",
    show_legends=True,
    legend_ncol=1,
    legend_bbox_to_anchor=(1.02, 0.5),
    fallback_palette="Set2",
    show=True,
):

    order = res["subset_order"]
    mat = res["mean_usage"].loc[order].copy()

    if zscore_rows:
        mat = mat.sub(mat.mean(axis=1), axis=0).div(
            mat.std(axis=1).replace(0, np.nan), axis=0
        ).fillna(0)
        cbar_label = heatmap_cbar_label
    else:
        cbar_label = "Mean program usage"

    obs_cols = [subset_col] + list(anno_cols)
    meta = adata.obs[obs_cols].drop_duplicates().copy()

    dup_check = meta.groupby(subset_col).size()
    if (dup_check > 1).any():
        bad = dup_check[dup_check > 1].index.tolist()[:10]
        raise ValueError(
            f"Some {subset_col} map to multiple annotation rows, e.g. {bad}. "
            f"Please ensure one-to-one mapping."
        )

    meta = meta.set_index(subset_col).loc[order]
    color_map = {} if color_map is None else color_map.copy()

    anno_color_tables = {}
    anno_rgb = {}

    for col in anno_cols:
        vals = meta[col].astype(str).fillna("NA")
        categories = list(pd.Index(vals.unique()))

        lut = {}
        missing = []

        for cat in categories:
            if cat in color_map:
                lut[cat] = color_map[cat]
            else:
                missing.append(cat)

        if len(missing) > 0:
            extra_colors = sns.color_palette(
                fallback_palette, n_colors=len(missing)
            ).as_hex()
            for cat, clr in zip(missing, extra_colors):
                lut[cat] = clr

        anno_color_tables[col] = lut
        anno_rgb[col] = vals.map(lut)

    n_anno = len(anno_cols)
    fig = plt.figure(figsize=figsize)

    gs = fig.add_gridspec(
        nrows=1,
        ncols=n_anno + 2,
        width_ratios=[0.35] * n_anno + [8, 0.35],
        wspace=0.05
    )

    anno_axes = [fig.add_subplot(gs[0, i]) for i in range(n_anno)]
    ax_hm = fig.add_subplot(gs[0, n_anno])
    cax = fig.add_subplot(gs[0, n_anno + 1])

    for i, col in enumerate(anno_cols):
        ax = anno_axes[i]
        rgb_list = np.array([mpl.colors.to_rgb(x) for x in anno_rgb[col]])
        rgb_img = rgb_list.reshape(len(order), 1, 3)

        ax.imshow(rgb_img, aspect="auto", interpolation="nearest", origin="upper")
        ax.set_xticks([0])
        ax.set_xticklabels([col], rotation=90, fontsize=10)
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(length=0)

    sns.heatmap(
        mat,
        ax=ax_hm,
        cmap=cmap,
        center=center if zscore_rows else None,
        cbar=True,
        cbar_ax=cax,
        yticklabels=False,
        xticklabels=True,
        linewidths=0,
    )
    ax_hm.set_xlabel("Programs")
    ax_hm.set_ylabel("")
    for spine in ax_hm.spines.values():
        spine.set_visible(False)

    cax.set_ylabel(cbar_label)

    if show_legends:
        handles = []
        labels = []
        for col in anno_cols:
            for k, v in anno_color_tables[col].items():
                handles.append(mpl.patches.Patch(facecolor=v, edgecolor="none"))
                labels.append(f"{col}: {k}")

        fig.legend(
            handles,
            labels,
            loc="center left",
            bbox_to_anchor=legend_bbox_to_anchor,
            frameon=False,
            fontsize=9,
            ncol=legend_ncol,
        )

    if show:
        plt.show()

    return fig, ax_hm, anno_axes, cax


In [ ]:
sample_program_res = compute_subset_program_similarity(
    T_cnmf_tumor,
    subset_col="Sample_ID",
    program_prefix="Program_",
    n_programs=27,
    min_cells=20,
    metric="correlation",
    linkage_method="average",
    zscore_programs=True,
)

fig, ax_hm, anno_axes, cax =plot_program_usage_heatmap_with_annotations(
    sample_program_res,
    adata=T_cnmf_tumor,
    subset_col="Sample_ID",
    anno_cols=("Period_merged", "Disease"),
    figsize=(8, 8),
    color_map=meta_color_map,
    zscore_rows=True,
)

fig.savefig(
    'figures/Fig5h_sample_program_usage.svg',
    dpi=300,
    bbox_inches="tight"
)


In [ ]:
fig, ax, dendro = plot_subset_dendrogram(
    sample_program_res,
    figsize=(14, 5),
    palette="Set1",

    leaf_rotation=90,
    leaf_fontsize=10,
    save='figures/Fig5h_sample_program_dendrogram.pdf'
)


# FigS5i

In [ ]:
def prepare_ecosystem_composition_matrix(
    df,
    name="composition matrix",
    drop_all_zero=True,
):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")

    x = df.copy()

    x.index = x.index.astype(str)
    x.index.name = "Sample_ID"
    x.columns.name = None

    if x.index.duplicated().any():
        duplicated_samples = (
            x.index[x.index.duplicated()]
            .unique()
            .tolist()
        )
        raise ValueError(
            f"{name} contains duplicated sample IDs: "
            f"{duplicated_samples[:10]}"
        )

    x = x.apply(pd.to_numeric, errors="raise").astype(float)

    if x.isna().any().any():
        raise ValueError(f"{name} contains missing values.")

    if np.isinf(x.to_numpy()).any():
        raise ValueError(f"{name} contains infinite values.")

    if (x < 0).any().any():
        raise ValueError(f"{name} contains negative values.")

    if drop_all_zero:
        all_zero = x.columns[x.sum(axis=0) == 0]

        if len(all_zero) > 0:
            print(
                f"{name}: removing {len(all_zero)} all-zero features:"
            )
            print(all_zero.tolist())

            x = x.drop(columns=all_zero)

    if x.shape[1] < 2:
        raise ValueError(
            f"{name} must contain at least two non-zero features."
        )

    row_sums = x.sum(axis=1)

    if (row_sums <= 0).any():
        bad_samples = row_sums.index[row_sums <= 0].tolist()

        raise ValueError(
            f"{name} contains samples with row sum <= 0: "
            f"{bad_samples[:10]}"
        )

    x = x.div(row_sums, axis=0)

    return x

def transform_composition_matrix(
    df,
    name="composition matrix",
    delta=None,
    center_features=True,
    scale_features=False,
):
    normalized = prepare_ecosystem_composition_matrix(
        df,
        name=name,
    )

    clr_matrix = clr_transform(
        normalized,
        delta=delta,
    )

    transformed = clr_matrix.copy()

    if center_features:
        transformed = transformed - transformed.mean(axis=0)

    if scale_features:
        feature_sd = transformed.std(axis=0, ddof=0)

        variable_features = feature_sd > 1e-12

        removed = transformed.columns[~variable_features]

        if len(removed) > 0:
            print(
                f"{name}: removing {len(removed)} "
                "zero-variance features:"
            )
            print(removed.tolist())

        transformed = transformed.loc[:, variable_features]
        feature_sd = feature_sd.loc[variable_features]

        transformed = transformed.div(
            feature_sd,
            axis=1,
        )

    return normalized, clr_matrix, transformed

def rv2_coefficient_from_gram(
    gram_x,
    gram_y,
):
    numerator = np.sum(gram_x * gram_y)

    denominator = np.sqrt(
        np.sum(gram_x ** 2)
        * np.sum(gram_y ** 2)
    )

    if denominator == 0:
        return np.nan

    return numerator / denominator

def compare_ecosystem_matrices(
    comp_raw,
    sample_program_usage,
    n_permutations=9999,
    random_state=0,
    delta=None,
    scale_features=False,
):
    if n_permutations < 1:
        raise ValueError("n_permutations must be at least 1.")

    comp_input = comp_raw.copy()
    program_input = sample_program_usage.copy()

    comp_input.index = comp_input.index.astype(str)
    program_input.index = program_input.index.astype(str)

    common_samples = comp_input.index[
        comp_input.index.isin(program_input.index)
    ]

    if len(common_samples) < 5:
        raise ValueError(
            f"Only {len(common_samples)} common samples were found."
        )

    comp_input = comp_input.loc[common_samples]
    program_input = program_input.loc[common_samples]

    comp_normalized, comp_clr, comp_transformed = (
        transform_composition_matrix(
            comp_input,
            name="State proportion matrix",
            delta=delta,
            center_features=True,
            scale_features=scale_features,
        )
    )

    program_normalized, program_clr, program_transformed = (
        transform_composition_matrix(
            program_input,
            name="Program usage matrix",
            delta=delta,
            center_features=True,
            scale_features=scale_features,
        )
    )

    x = comp_transformed.to_numpy(dtype=float)
    y = program_transformed.to_numpy(dtype=float)

    gram_x = x @ x.T
    gram_y = y @ y.T

    np.fill_diagonal(gram_x, 0.0)
    np.fill_diagonal(gram_y, 0.0)

    observed_rv2 = rv2_coefficient_from_gram(
        gram_x,
        gram_y,
    )

    rng = np.random.default_rng(random_state)

    null_distribution = np.empty(
        n_permutations,
        dtype=float,
    )

    n_samples = len(common_samples)

    for i in range(n_permutations):
        permutation = rng.permutation(n_samples)

        gram_y_permuted = gram_y[
            np.ix_(permutation, permutation)
        ]

        null_distribution[i] = rv2_coefficient_from_gram(
            gram_x,
            gram_y_permuted,
        )

    permutation_p = (
        np.sum(null_distribution >= observed_rv2) + 1
    ) / (n_permutations + 1)

    result = pd.DataFrame(
        {
            "n_samples": [n_samples],
            "n_states": [comp_transformed.shape[1]],
            "n_programs": [program_transformed.shape[1]],
            "RV2": [observed_rv2],
            "permutation_p": [permutation_p],
            "scale_features": [scale_features],
        }
    )

    return {
        "result": result,
        "common_samples": common_samples,
        "comp_normalized": comp_normalized,
        "program_normalized": program_normalized,
        "comp_clr": comp_clr,
        "program_clr": program_clr,
        "comp_transformed": comp_transformed,
        "program_transformed": program_transformed,
        "gram_state": pd.DataFrame(
            gram_x,
            index=common_samples,
            columns=common_samples,
        ),
        "gram_program": pd.DataFrame(
            gram_y,
            index=common_samples,
            columns=common_samples,
        ),
        "null_distribution": null_distribution,
    }


In [ ]:
def plot_ecosystem_concordance(
    rv2_res,
    sample_meta=None,
    color_col=None,
    color_map=None,
    figsize=(16, 4.8),
    bins=35,
    pair_point_size=14,
    sample_point_size=65,
    pair_alpha=0.20,
    connection_alpha=0.25,
    annotate=False,
    save=None,
    dpi=300,
):

    required_keys = [
        "result",
        "common_samples",
        "gram_state",
        "gram_program",
        "comp_transformed",
        "program_transformed",
        "null_distribution",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in rv2_res
    ]

    if missing_keys:
        raise KeyError(
            f"rv2_res is missing keys: {missing_keys}"
        )

    result = rv2_res["result"]

    observed_rv2 = float(
        result.loc[result.index[0], "RV2"]
    )

    permutation_p = float(
        result.loc[result.index[0], "permutation_p"]
    )

    null_distribution = np.asarray(
        rv2_res["null_distribution"],
        dtype=float,
    )

    samples = pd.Index(
        rv2_res["common_samples"]
    ).astype(str)

    n_samples = len(samples)

    gram_state = rv2_res["gram_state"].copy()
    gram_program = rv2_res["gram_program"].copy()

    gram_state.index = gram_state.index.astype(str)
    gram_state.columns = gram_state.columns.astype(str)

    gram_program.index = gram_program.index.astype(str)
    gram_program.columns = gram_program.columns.astype(str)

    gram_state = gram_state.loc[samples, samples]
    gram_program = gram_program.loc[samples, samples]

    upper_triangle = np.triu_indices(
        n_samples,
        k=1,
    )

    pairwise_df = pd.DataFrame(
        {
            "State similarity": (
                gram_state.to_numpy()[upper_triangle]
            ),
            "Program similarity": (
                gram_program.to_numpy()[upper_triangle]
            ),
        }
    )

    comp_transformed = (
        rv2_res["comp_transformed"]
        .copy()
    )

    program_transformed = (
        rv2_res["program_transformed"]
        .copy()
    )

    comp_transformed.index = (
        comp_transformed.index.astype(str)
    )

    program_transformed.index = (
        program_transformed.index.astype(str)
    )

    X_state = comp_transformed.loc[
        samples
    ].to_numpy(dtype=float)

    X_program = program_transformed.loc[
        samples
    ].to_numpy(dtype=float)

    state_pca = PCA(n_components=2).fit_transform(
        X_state
    )

    program_pca = PCA(n_components=2).fit_transform(
        X_program
    )

    state_aligned, program_aligned, disparity = (
        procrustes(
            state_pca,
            program_pca,
        )
    )

    procrustes_state = pd.DataFrame(
        state_aligned,
        index=samples,
        columns=["Dimension 1", "Dimension 2"],
    )

    procrustes_program = pd.DataFrame(
        program_aligned,
        index=samples,
        columns=["Dimension 1", "Dimension 2"],
    )

    if sample_meta is not None:
        meta = sample_meta.copy()
        meta.index = meta.index.astype(str)
        meta = meta.reindex(samples)
    else:
        meta = pd.DataFrame(index=samples)

    if color_col is not None:
        if color_col not in meta.columns:
            raise KeyError(
                f"{color_col!r} not found in sample_meta."
            )

        present_levels = list(
            pd.unique(meta[color_col].dropna())
        )

        if color_map is None:
            palette_values = sns.color_palette(
                "tab20",
                n_colors=max(len(present_levels), 1),
            )

            palette = dict(
                zip(
                    present_levels,
                    palette_values,
                )
            )
        else:
            palette = color_map.copy()

        point_colors = [
            palette.get(value, "0.65")
            if pd.notna(value)
            else "0.65"
            for value in meta[color_col]
        ]
    else:
        present_levels = []
        palette = {}
        point_colors = ["C0"] * n_samples

    fig, axes = plt.subplots(
        1,
        3,
        figsize=figsize,
    )

    axes[0].hist(
        null_distribution,
        bins=bins,
        color="0.45",
        edgecolor="white",
        linewidth=0.5,
    )

    axes[0].axvline(
        observed_rv2,
        color="black",
        linewidth=2,
        label=f"Observed RV2 = {observed_rv2:.3f}",
    )

    axes[0].set_xlabel(
        "RV2 under sample-label permutation"
    )

    axes[0].set_ylabel(
        "Permutation count"
    )

    axes[0].set_title(
        "Permutation test"
    )

    axes[0].text(
        0.98,
        0.95,
        f"RV2 = {observed_rv2:.3f}\n"
        f"P = {permutation_p:.1e}",
        transform=axes[0].transAxes,
        ha="right",
        va="top",
    )

    axes[1].scatter(
        pairwise_df["State similarity"],
        pairwise_df["Program similarity"],
        s=pair_point_size,
        alpha=pair_alpha,
        linewidth=0,
    )

    x_values = pairwise_df[
        "State similarity"
    ].to_numpy()

    y_values = pairwise_df[
        "Program similarity"
    ].to_numpy()

    if (
        np.std(x_values) > 0
        and np.std(y_values) > 0
    ):
        slope, intercept = np.polyfit(
            x_values,
            y_values,
            deg=1,
        )

        x_line = np.linspace(
            x_values.min(),
            x_values.max(),
            200,
        )

        axes[1].plot(
            x_line,
            slope * x_line + intercept,
            color="black",
            linewidth=1.5,
        )

    axes[1].set_xlabel(
        "State-composition similarity"
    )

    axes[1].set_ylabel(
        "Program-usage similarity"
    )

    axes[1].set_title(
        "Pairwise sample relationships"
    )

    axes[1].text(
        0.04,
        0.96,
        f"RV2 = {observed_rv2:.3f}\n"
        f"{len(pairwise_df):,} sample pairs",
        transform=axes[1].transAxes,
        ha="left",
        va="top",
    )

    for i, sample_id in enumerate(samples):
        axes[2].plot(
            [
                state_aligned[i, 0],
                program_aligned[i, 0],
            ],
            [
                state_aligned[i, 1],
                program_aligned[i, 1],
            ],
            color="0.65",
            linewidth=0.8,
            alpha=connection_alpha,
            zorder=1,
        )

    axes[2].scatter(
        state_aligned[:, 0],
        state_aligned[:, 1],
        c=point_colors,
        s=sample_point_size,
        marker="o",
        linewidth=0,
        zorder=2,
    )

    axes[2].scatter(
        program_aligned[:, 0],
        program_aligned[:, 1],
        c=point_colors,
        s=sample_point_size,
        marker="X",
        linewidth=0,
        zorder=3,
    )

    if annotate:
        for i, sample_id in enumerate(samples):
            axes[2].text(
                program_aligned[i, 0],
                program_aligned[i, 1],
                sample_id,
                fontsize=7,
            )

    axes[2].set_xlabel(
        "Procrustes dimension 1"
    )

    axes[2].set_ylabel(
        "Procrustes dimension 2"
    )

    axes[2].set_title(
        f"Aligned sample embeddings\n"
        f"disparity = {disparity:.3f}"
    )

    axes[2].set_aspect(
        "equal",
        adjustable="datalim",
    )

    representation_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="none",
            markerfacecolor="0.35",
            markeredgewidth=0,
            markersize=7,
            label="State composition",
        ),
        Line2D(
            [0],
            [0],
            marker="X",
            linestyle="none",
            markerfacecolor="0.35",
            markeredgewidth=0,
            markersize=7,
            label="Program usage",
        ),
    ]

    representation_legend = axes[2].legend(
        handles=representation_handles,
        loc="lower left",
        frameon=False,
    )

    axes[2].add_artist(
        representation_legend
    )

    if color_col is not None:
        color_handles = [
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="none",
                markerfacecolor=palette.get(
                    level,
                    "0.65",
                ),
                markeredgewidth=0,
                markersize=6,
                label=str(level),
            )
            for level in present_levels
        ]

        axes[2].legend(
            handles=color_handles,
            title=color_col,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
        )

    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(length=3)

    plt.tight_layout()

    if save is not None:
        fig.savefig(
            save,
            dpi=dpi,
            bbox_inches="tight",
            transparent=True,
        )

    plt.show()

    plot_data = {
        "pairwise_similarity": pairwise_df,
        "procrustes_state": procrustes_state,
        "procrustes_program": procrustes_program,
        "procrustes_disparity": disparity,
    }

    return fig, axes, plot_data


In [ ]:
program_cols = [f"Program_{i}" for i in range(1, 28)]

missing_cols = [col for col in program_cols if col not in T_cnmf.obs.columns]
if missing_cols:
    raise KeyError(f"Missing program columns: {missing_cols}")

cell_program_sum = T_cnmf_tumor.obs[program_cols].sum(axis=1)

print(cell_program_sum.describe())
print(
    "All cells sum to 1:",
    np.allclose(cell_program_sum, 1.0, atol=1e-6)
)

sample_n_cells = (
    T_cnmf_tumor.obs.groupby("Sample_ID", observed=True)
    .size()
    .rename("n_cells")
)

sample_program_usage_27 = (
    T_cnmf_tumor.obs.groupby("Sample_ID", observed=True)[program_cols]
    .mean()
)

min_cells_per_sample = 20

valid_samples = sample_n_cells[
    sample_n_cells >= min_cells_per_sample
].index

sample_program_usage_27 = sample_program_usage_27.loc[valid_samples]
sample_n_cells = sample_n_cells.loc[valid_samples]

sample_program_sum = sample_program_usage_27.sum(axis=1)

print(sample_program_usage_27.shape)
print(sample_program_sum.describe())
print(
    "All samples sum to 1:",
    np.allclose(sample_program_sum, 1.0, atol=1e-6)
)


In [ ]:
comp_raw = res_composition["comp_raw"]


In [ ]:
selected_programs = [
    "Program_1",
    "Program_3",
    "Program_4",
    "Program_7",
    "Program_8",
    "Program_12",
    "Program_13",
    "Program_14",
    "Program_16",
    "Program_18",
    "Program_19",
    "Program_20",
    "Program_22",
    "Program_27"

]


In [ ]:
rv2_res_27 = compare_ecosystem_matrices(
    comp_raw=comp_raw,
    sample_program_usage=sample_program_usage_27,
    n_permutations=9999,
    random_state=0,
    scale_features=False,
)


In [ ]:
sample_program_usage_14 = sample_program_usage_27[selected_programs].copy()

rv2_res_14 = compare_ecosystem_matrices(
    comp_raw=comp_raw,
    sample_program_usage=sample_program_usage_14,
    n_permutations=9999,
    random_state=0,
    scale_features=False,
)


In [ ]:
rv2_27 = rv2_res_27["result"].loc[0, "RV2"]
p_27 = rv2_res_27["result"].loc[0, "permutation_p"]

rv2_14 = rv2_res_14["result"].loc[0, "RV2"]
p_14 = rv2_res_14["result"].loc[0, "permutation_p"]

rv2_retention = rv2_14 / rv2_27
rv2_difference = rv2_14 - rv2_27

comparison = pd.DataFrame({
    "program_set": ["All 27 programs", "Selected 14 programs"],
    "n_programs": [27, 14],
    "RV2": [rv2_27, rv2_14],
    "permutation_p": [p_27, p_14],
})

print(comparison)

print(f"\nAbsolute difference: {rv2_difference:.4f}")
print(f"RV2 retention: {rv2_retention:.1%}")


In [ ]:
def plot_two_rv2_permutation_tests(
    rv2_res_27,
    rv2_res_14,
    labels=("All 27 programs", "Selected 14 programs"),
    bins=40,
    figsize=(10, 4.2),
    hist_color="0.50",
    observed_color="black",
    save=None,
    dpi=300,
):

    results = [rv2_res_27, rv2_res_14]

    null_distributions = [
        np.asarray(res["null_distribution"], dtype=float)
        for res in results
    ]

    observed_rv2 = [
        float(res["result"].iloc[0]["RV2"])
        for res in results
    ]

    permutation_p = [
        float(res["result"].iloc[0]["permutation_p"])
        for res in results
    ]

    combined_null = np.concatenate(null_distributions)

    null_min = combined_null.min()
    null_max = combined_null.max()
    observed_max = max(observed_rv2)

    x_range = observed_max - min(null_min, 0)
    x_padding = max(x_range * 0.04, 0.01)

    x_min = min(null_min, 0) - x_padding
    x_max = observed_max + x_padding

    bin_edges = np.linspace(
        combined_null.min(),
        combined_null.max(),
        bins + 1,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=figsize,
        sharex=True,
        sharey=True,
    )

    for ax, null_dist, rv2, p_value, panel_title in zip(
        axes,
        null_distributions,
        observed_rv2,
        permutation_p,
        labels,
    ):
        ax.hist(
            null_dist,
            bins=bin_edges,
            color=hist_color,
            edgecolor="white",
            linewidth=0.4,
        )

        ax.axvline(
            rv2,
            color=observed_color,
            linewidth=2,
        )

        ax.text(
            0.97,
            0.95,
            f"RV2 = {rv2:.3f}\n"
            f"$P$ = {p_value:.1e}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=10,
        )

        ax.set_title(panel_title)
        ax.set_xlabel(
            "RV2 under sample-label permutation"
        )

        ax.set_xlim(x_min, x_max)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(length=3)

    axes[0].set_ylabel("Permutation count")

    plt.tight_layout()

    if save is not None:
        fig.savefig(
            save,
            dpi=dpi,
            bbox_inches="tight",
            transparent=True,
        )

    plt.show()

    return fig, axes


In [ ]:
fig, axes = plot_two_rv2_permutation_tests(
    rv2_res_27=rv2_res_27,
    rv2_res_14=rv2_res_14,
    labels=(
        "All 27 programs",
        "Selected 14 programs",
    ),
    bins=40,
    figsize=(9, 4),
    hist_color="0.35",
    save=(
        'figures/FigS5i_RV2_permutation.pdf'
    ),
)


# FigS5e

In [ ]:
cols = ["Sample_ID", "Period_merged", "Disease", "Sample_Type"]

for col in cols:
    if col not in T_cnmf.obs.columns:
        raise ValueError(f"T_cnmf.obs['{col}'] 不存在，请检查字段名")

T_cnmf.obs["Period_Disease_Type"] = (
    T_cnmf.obs["Period_merged"].astype(str) + "_" +
    T_cnmf.obs["Disease"].astype(str) + "_" +
    T_cnmf.obs["Sample_Type"].astype(str)
)

cpdb_sample_meta = (
    T_cnmf.obs[
        ["Sample_ID", "Period_merged", "Disease", "Sample_Type", "Period_Disease_Type"]
    ]
    .drop_duplicates()
    .copy()
)

check_sample_meta = (
    cpdb_sample_meta
    .groupby("Sample_ID")
    .agg(
        n_Period_merged=("Period_merged", "nunique"),
        n_Disease=("Disease", "nunique"),
        n_Sample_Type=("Sample_Type", "nunique"),
        n_Period_Disease_Type=("Period_Disease_Type", "nunique")
    )
    .reset_index()
)

problem_samples = check_sample_meta[
    (check_sample_meta["n_Period_merged"] > 1) |
    (check_sample_meta["n_Disease"] > 1) |
    (check_sample_meta["n_Sample_Type"] > 1) |
    (check_sample_meta["n_Period_Disease_Type"] > 1)
]

print("存在分组不唯一的样本数:", problem_samples.shape[0])

if problem_samples.shape[0] > 0:
    print(problem_samples.head())
    raise ValueError('有 Sample_ID 对应多个分组信息，请先检查 T_cnmf.obs')

cpdb_sample_meta = cpdb_sample_meta.drop_duplicates(subset=["Sample_ID"]).copy()

print("样本信息矩阵维度:", cpdb_sample_meta.shape)
cpdb_sample_meta.head()


In [ ]:
cpdb_T = T_cnmf

sample_col = "Sample_ID"
celltype_col = "T_sub_level_3"
group_col = "Period_Disease_Type"
layer_key = "raw.X.log_norm"

base_outdir = 'results/cpdb/samples'
os.makedirs(base_outdir, exist_ok=True)

min_cells_per_celltype = 10

min_celltypes_per_sample = 2

required_cols = [sample_col, celltype_col]
for col in required_cols:
    if col not in cpdb_T.obs.columns:
        raise ValueError(f"adata.obs['{col}'] 不存在，请检查")

if layer_key not in cpdb_T.layers:
    raise ValueError(f"adata.layers['{layer_key}'] 不存在，请检查 layer 名称")

print("样本数:", cpdb_T.obs[sample_col].nunique())
print("细胞类型数:", cpdb_T.obs[celltype_col].nunique())
print("使用 layer:", layer_key)

sample_list = cpdb_T.obs[sample_col].dropna().unique().tolist()

exported_samples = []
skipped_samples = []

for sample in sample_list:
    print(f"\nProcessing sample: {sample}")

    adata_s = cpdb_T[cpdb_T.obs[sample_col] == sample].copy()

    adata_s = adata_s[adata_s.obs[celltype_col].notna()].copy()
    adata_s = adata_s[adata_s.obs[celltype_col].astype(str) != ""].copy()

    ct_counts = adata_s.obs[celltype_col].value_counts()
    keep_celltypes = ct_counts[ct_counts >= min_cells_per_celltype].index.tolist()
    adata_s = adata_s[adata_s.obs[celltype_col].isin(keep_celltypes)].copy()

    if adata_s.obs[celltype_col].nunique() < min_celltypes_per_sample:
        print(f"跳过 {sample}: 剩余细胞类型少于 {min_celltypes_per_sample}")
        skipped_samples.append(sample)
        continue

    adata_s.obs_names = adata_s.obs_names.str.replace("-", "_", regex=False)

    if adata_s.obs_names.duplicated().any():
        print(f"跳过 {sample}: barcode 替换后出现重复")
        skipped_samples.append(sample)
        continue

    sample_outdir = os.path.join(base_outdir, str(sample))
    os.makedirs(sample_outdir, exist_ok=True)

    counts_file = os.path.join(sample_outdir, "cellphonedb_counts.h5ad")
    meta_file = os.path.join(sample_outdir, "cellphonedb_meta.csv")

    counts_data = sc.AnnData(
        X=adata_s.layers[layer_key],
        obs=pd.DataFrame(index=adata_s.obs_names),
        var=pd.DataFrame(index=adata_s.var_names)
    )

    counts_data.write_h5ad(counts_file)

    meta_df = adata_s.obs[[celltype_col]].copy()
    meta_df = meta_df.rename(columns={celltype_col: "cell_type"})
    meta_df.index.name = "Cell"
    meta_df.to_csv(meta_file)

    print(f"导出完成: {sample}")
    print("细胞数:", adata_s.n_obs)
    print("细胞类型数:", adata_s.obs[celltype_col].nunique())

    exported_samples.append(sample)

print("\n===============================")
print("导出完成样本数:", len(exported_samples))
print("跳过样本数:", len(skipped_samples))
print("跳过样本:", skipped_samples)


In [ ]:
base_dir = 'results/cpdb/samples'

summary_out_file = os.path.join(
    base_dir,
    "merged_results",
    "sample_meta_simplification_summary.csv"
)

merged_meta_out_file = os.path.join(
    base_dir,
    "merged_results",
    "all_samples_cellphonedb_meta_simplified_merged.csv"
)

celltype_count_out_file = os.path.join(
    base_dir,
    "merged_results",
    "all_samples_simplified_celltype_counts.csv"
)

os.makedirs(os.path.join(base_dir, "merged_results"), exist_ok=True)

sample_dirs = [
    d for d in sorted(os.listdir(base_dir))
    if os.path.isdir(os.path.join(base_dir, d))
]

print(f"检测到样本目录数: {len(sample_dirs)}")

processed_samples = []
skipped_samples = []

summary_list = []
merged_meta_list = []

for sample in sample_dirs:
    sample_dir = os.path.join(base_dir, sample)

    meta_file = os.path.join(sample_dir, "cellphonedb_meta.csv")
    out_file = os.path.join(sample_dir, "cellphonedb_meta_simplified.csv")

    if not os.path.exists(meta_file):
        print(f"[跳过] {sample}: 未找到 cellphonedb_meta.csv")
        skipped_samples.append(sample)
        continue

    try:
        meta = pd.read_csv(meta_file)

        if meta.shape[1] < 2:
            print(f"[跳过] {sample}: meta 文件列数少于 2")
            skipped_samples.append(sample)
            continue

        cell_col = meta.columns[0]
        celltype_col = meta.columns[1]

        original_celltypes = sorted(meta[celltype_col].astype(str).unique())
        n_celltypes_before = len(original_celltypes)

        meta_simplified = meta.copy()
        meta_simplified[celltype_col] = (
            meta_simplified[celltype_col]
            .astype(str)
            .str.replace(r"_c\d+$", "", regex=True)
        )

        simplified_celltypes = sorted(meta_simplified[celltype_col].astype(str).unique())
        n_celltypes_after = len(simplified_celltypes)

        meta_simplified.to_csv(out_file, index=False)

        meta_simplified_for_merge = meta_simplified.copy()
        meta_simplified_for_merge["sample"] = sample

        meta_simplified_for_merge = meta_simplified_for_merge.rename(
            columns={
                cell_col: "Cell",
                celltype_col: "cell_type_simplified"
            }
        )

        merged_meta_list.append(meta_simplified_for_merge)

        summary_list.append({
            "sample": sample,
            "n_cells": meta.shape[0],
            "celltype_col": celltype_col,
            "n_celltypes_before": n_celltypes_before,
            "n_celltypes_after": n_celltypes_after,
            "celltypes_before": ";".join(original_celltypes),
            "celltypes_after": ";".join(simplified_celltypes),
            "meta_file": meta_file,
            "simplified_meta_file": out_file
        })

        print(
            f"[完成] {sample}: cells={meta.shape[0]}, "
            f"cell types {n_celltypes_before} -> {n_celltypes_after}"
        )

        processed_samples.append(sample)

    except Exception as e:
        print(f"[失败] {sample}: {e}")
        skipped_samples.append(sample)

summary_df = pd.DataFrame(summary_list)

summary_df.to_csv(summary_out_file, index=False)

print("\n每个样本简化统计已保存:")
print(summary_out_file)

if len(merged_meta_list) == 0:
    raise RuntimeError("没有成功处理任何 meta 文件，请检查路径")

merged_meta_simplified = pd.concat(
    merged_meta_list,
    axis=0,
    ignore_index=True
)

merged_meta_simplified.to_csv(merged_meta_out_file, index=False)

print("\n合并后的 simplified meta 已保存:")
print(merged_meta_out_file)

print("\n合并后 meta 维度:")
print(merged_meta_simplified.shape)

all_celltypes = sorted(
    merged_meta_simplified["cell_type_simplified"].astype(str).unique()
)

print("\n合并后总体细胞类型数量:")
print(len(all_celltypes))

print("\n合并后总体细胞类型列表:")
for ct in all_celltypes:
    print(ct)

celltype_count_df = (
    merged_meta_simplified
    .groupby("cell_type_simplified")
    .agg(
        n_cells=("Cell", "count"),
        n_samples=("sample", "nunique")
    )
    .reset_index()
    .sort_values(["n_cells"], ascending=False)
)

celltype_count_df.to_csv(celltype_count_out_file, index=False)

print("\n合并后细胞类型分布已保存:")
print(celltype_count_out_file)

print("\n合并后细胞类型分布前 30:")
print(celltype_count_df.head(30).to_string(index=False))

print("\n===============================")
print("处理完成")
print("===============================")
print("成功样本数:", len(processed_samples))
print("跳过/失败样本数:", len(skipped_samples))

if len(skipped_samples) > 0:
    print("跳过/失败样本:")
    print(skipped_samples)

print("\n简化前每个样本细胞类型数概览:")
print(summary_df[["sample", "n_cells", "n_celltypes_before", "n_celltypes_after"]].head(20).to_string(index=False))

print("\n合并后总体细胞类型数量:", len(all_celltypes))


In [ ]:
cpdb_file_path = 'cellphonedb.zip'
base_outdir = 'results/cpdb/samples'

sample_list = [
    s for s in os.listdir(base_outdir)
    if os.path.isdir(os.path.join(base_outdir, s))
]

for sample in sample_list:
    print(f"\nRunning CellPhoneDB for sample: {sample}")

    sample_dir = os.path.join(base_outdir, sample)

    meta_file_path = os.path.join(sample_dir, "cellphonedb_meta.csv")
    counts_file_path = os.path.join(sample_dir, "cellphonedb_counts.h5ad")
    output_path = os.path.join(sample_dir, "cellphoneDB")

    if not os.path.exists(meta_file_path):
        print(f"跳过 {sample}: meta 文件不存在")
        continue

    if not os.path.exists(counts_file_path):
        print(f"跳过 {sample}: counts 文件不存在")
        continue

    try:
        cpdb_results = cpdb_statistical_analysis_method.call(
            cpdb_file_path=cpdb_file_path,
            meta_file_path=meta_file_path,
            counts_file_path=counts_file_path,
            counts_data="hgnc_symbol",
            threshold=0.1,
            iterations=1000,
            pvalue=0.05,
            threads=5,
            debug_seed=123,
            score_interactions=True,
            output_path=output_path
        )

        print(f"完成: {sample}")

    except Exception as e:
        print(f"样本 {sample} 运行失败:")
        print(e)


In [ ]:
base_dir = 'results/cpdb/samples'

out_dir = os.path.join(base_dir, "merged_results")
os.makedirs(out_dir, exist_ok=True)

out_long_file = os.path.join(out_dir, "all_samples_cpdb_lr_level_long.csv")

id_cols = [
    "id_cp_interaction",
    "interacting_pair",
    "partner_a",
    "partner_b",
    "gene_a",
    "gene_b",
    "secreted",
    "receptor_a",
    "receptor_b",
    "annotation_strategy",
    "is_integrin",
    "directionality",
    "classification"
]

def find_one_file(cpdb_dir, pattern):
    files = glob.glob(os.path.join(cpdb_dir, pattern))
    if len(files) == 0:
        return None

    files = sorted(files, key=os.path.getmtime, reverse=True)
    return files[0]

def read_cpdb_table(file_path):
    cols = pd.read_csv(
        file_path,
        sep="\t",
        nrows=0
    ).columns.tolist()

    dtype_dict = {
        col: "string"
        for col in id_cols
        if col in cols
    }

    df = pd.read_csv(
        file_path,
        sep="\t",
        dtype=dtype_dict,
        low_memory=False
    )

    for col in ["classification", "secreted", "receptor_a", "receptor_b", "directionality"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    if "classification" in df.columns:
        df["classification"] = df["classification"].replace("", "Unclassified")

    return df

def cpdb_wide_to_long(df, value_name):
    existing_id_cols = [c for c in id_cols if c in df.columns]

    cellpair_cols = [
        c for c in df.columns
        if c not in existing_id_cols and "|" in c
    ]

    if len(cellpair_cols) == 0:
        raise ValueError("没有识别到 cellpair 列，请检查 CellPhoneDB 输出文件列名")

    long_df = df.melt(
        id_vars=existing_id_cols,
        value_vars=cellpair_cols,
        var_name="cellpair",
        value_name=value_name
    )

    return long_df

def as_bool_series(x):
    return x.astype(str).str.lower().isin(["true", "1", "yes"])

all_sample_long = []
failed_samples = []

sample_dirs = [
    d for d in sorted(os.listdir(base_dir))
    if os.path.isdir(os.path.join(base_dir, d))
]

print(f"检测到样本目录数: {len(sample_dirs)}")

for sample in sample_dirs:
    sample_dir = os.path.join(base_dir, sample)
    cpdb_dir = os.path.join(sample_dir, 'cellphoneDB')

    if not os.path.isdir(cpdb_dir):
        print(f"[跳过] {sample}: 未找到 cellphoneDB 子目录")
        continue

    means_file = find_one_file(cpdb_dir, "statistical_analysis_means_*.txt")
    pvals_file = find_one_file(cpdb_dir, "statistical_analysis_pvalues_*.txt")
    sig_file = find_one_file(cpdb_dir, "statistical_analysis_significant_means_*.txt")
    scores_file = find_one_file(cpdb_dir, "statistical_analysis_interaction_scores_*.txt")

    if means_file is None or pvals_file is None:
        print(f"[跳过] {sample}: 缺少 means 或 pvalues 文件")
        failed_samples.append(sample)
        continue

    print(f"[读取] {sample}")

    try:

        means = read_cpdb_table(means_file)
        pvals = read_cpdb_table(pvals_file)

        means_long = cpdb_wide_to_long(means, "mean")
        pvals_long = cpdb_wide_to_long(pvals, "pvalue")

        merge_keys = ["id_cp_interaction", "interacting_pair", "cellpair"]
        merge_keys = [
            k for k in merge_keys
            if k in means_long.columns and k in pvals_long.columns
        ]

        sample_long = means_long.merge(
            pvals_long[merge_keys + ["pvalue"]],
            on=merge_keys,
            how="left"
        )

        if sig_file is not None:
            sig = read_cpdb_table(sig_file)
            sig_long = cpdb_wide_to_long(sig, "significant_mean")

            sample_long = sample_long.merge(
                sig_long[merge_keys + ["significant_mean"]],
                on=merge_keys,
                how="left"
            )
        else:
            sample_long["significant_mean"] = np.nan

        if scores_file is not None:
            scores = read_cpdb_table(scores_file)
            scores_long = cpdb_wide_to_long(scores, "interaction_score")

            sample_long = sample_long.merge(
                scores_long[merge_keys + ["interaction_score"]],
                on=merge_keys,
                how="left"
            )
        else:
            sample_long["interaction_score"] = np.nan

        sample_long["sample"] = sample

        cellpair_split = sample_long["cellpair"].str.split("|", n=1, expand=True)
        sample_long["source"] = cellpair_split[0]
        sample_long["target"] = cellpair_split[1]

        for col in ["mean", "pvalue", "significant_mean", "interaction_score"]:
            if col in sample_long.columns:
                sample_long[col] = pd.to_numeric(sample_long[col], errors="coerce")

        sample_long["is_significant"] = sample_long["pvalue"] < 0.05

        if "secreted" in sample_long.columns:
            sample_long["secreted_flag"] = as_bool_series(sample_long["secreted"])
        else:
            sample_long["secreted_flag"] = False

        if "receptor_a" in sample_long.columns:
            sample_long["receptor_a_flag"] = as_bool_series(sample_long["receptor_a"])
        else:
            sample_long["receptor_a_flag"] = False

        if "receptor_b" in sample_long.columns:
            sample_long["receptor_b_flag"] = as_bool_series(sample_long["receptor_b"])
        else:
            sample_long["receptor_b_flag"] = False

        if "classification" in sample_long.columns:
            sample_long["classification"] = (
                sample_long["classification"]
                .fillna("Unclassified")
                .astype(str)
                .replace("", "Unclassified")
            )

        sample_long["secretor"] = np.select(
            [
                sample_long["receptor_b_flag"],
                sample_long["receptor_a_flag"]
            ],
            [
                sample_long["source"],
                sample_long["target"]
            ],
            default=sample_long["source"]
        )

        sample_long["receiver"] = np.select(
            [
                sample_long["receptor_b_flag"],
                sample_long["receptor_a_flag"]
            ],
            [
                sample_long["target"],
                sample_long["source"]
            ],
            default=sample_long["target"]
        )

        first_cols = [
            "sample",
            "source",
            "target",
            "secretor",
            "receiver",
            "cellpair",
            "id_cp_interaction",
            "interacting_pair",
            "partner_a",
            "partner_b",
            "gene_a",
            "gene_b",
            "secreted",
            "secreted_flag",
            "receptor_a",
            "receptor_b",
            "receptor_a_flag",
            "receptor_b_flag",
            "classification",
            "directionality",
            "mean",
            "pvalue",
            "significant_mean",
            "interaction_score",
            "is_significant"
        ]

        existing_first_cols = [c for c in first_cols if c in sample_long.columns]
        other_cols = [c for c in sample_long.columns if c not in existing_first_cols]
        sample_long = sample_long[existing_first_cols + other_cols]

        all_sample_long.append(sample_long)

    except Exception as e:
        print(f"[失败] {sample}: {e}")
        failed_samples.append(sample)

if len(all_sample_long) == 0:
    raise RuntimeError("没有成功读取任何样本，请检查路径和文件格式")

all_cpdb_long = pd.concat(all_sample_long, axis=0, ignore_index=True)
all_cpdb_long = all_cpdb_long.merge(
    cpdb_sample_meta,
    left_on="sample",
    right_on="Sample_ID",
    how="left",
    validate="many_to_one",
)

print("合并完成")
print("总行数:", all_cpdb_long.shape[0])
print("总列数:", all_cpdb_long.shape[1])
print("样本数:", all_cpdb_long["sample"].nunique())

if "cellpair" in all_cpdb_long.columns:
    bad_cellpair = all_cpdb_long.loc[~all_cpdb_long["cellpair"].str.contains(r"\|", regex=True, na=False), "cellpair"].unique()
    print("异常 cellpair 数量:", len(bad_cellpair))
    if len(bad_cellpair) > 0:
        print("异常 cellpair 示例:", bad_cellpair[:10])

if "secretor" in all_cpdb_long.columns:
    n_rank_secretor = (all_cpdb_long["secretor"] == "rank").sum()
    print("secretor == 'rank' 行数:", n_rank_secretor)

all_cpdb_long.to_csv(out_long_file, index=False)

print("保存文件:", out_long_file)

if len(failed_samples) > 0:
    failed_file = os.path.join(out_dir, "failed_samples.txt")
    with open(failed_file, "w") as f:
        for s in failed_samples:
            f.write(s + "\n")
    print("失败样本已保存:", failed_file)


In [ ]:
file_path = 'results/cpdb/samples/merged_results/all_samples_cpdb_lr_level_long.csv'

all_cpdb_long = pd.read_csv(file_path, low_memory=False)


In [ ]:
print("原始表格维度:", all_cpdb_long.shape)
print(all_cpdb_long["is_significant"].value_counts(dropna=False))
out_file = 'results/cpdb/samples/merged_results/all_samples_cpdb_lr_level_significant.csv'

sig_cpdb_long = all_cpdb_long[
    all_cpdb_long["is_significant"].astype(str).str.lower().isin(["true", "1"])
].copy()

print("显著互作表格维度:", sig_cpdb_long.shape)

sig_cpdb_long.to_csv(out_file, index=False)

print("已保存:", out_file)


In [ ]:
file_path = 'results/cpdb/samples/merged_results/all_samples_cpdb_lr_level_significant.csv'

all_cpdb_sig = pd.read_csv(file_path, low_memory=False)


In [ ]:
sample_cpdb_summary = (
    all_cpdb_sig
    .groupby("sample")
    .agg(
        n_significant_interactions=("interacting_pair", "count"),
        n_unique_lr=("interacting_pair", "nunique"),
        n_cellpairs=("cellpair", "nunique"),
        total_significant_strength=("significant_mean", "sum"),
        mean_significant_strength=("significant_mean", "mean"),
        total_interaction_score=("interaction_score", "sum"),
        mean_interaction_score=("interaction_score", "mean")
    )
    .reset_index()
)

meta_cols = [
    "sample",
    "Sample_ID",
    "Period_merged",
    "Disease",
    "Sample_Type",
    "Period_Disease_Type"
]

existing_meta_cols = [c for c in meta_cols if c in all_cpdb_sig.columns]

sample_meta_for_merge = (
    all_cpdb_sig[existing_meta_cols]
    .drop_duplicates(subset=["sample"])
)

sample_cpdb_summary = sample_cpdb_summary.merge(
    sample_meta_for_merge,
    on="sample",
    how="left"
)

first_cols = [
    "sample",
    "Sample_ID",
    "Period_merged",
    "Disease",
    "Sample_Type",
    "Period_Disease_Type",
    "n_significant_interactions",
    "n_unique_lr",
    "n_cellpairs",
    "total_significant_strength",
    "mean_significant_strength",
    "total_interaction_score",
    "mean_interaction_score"
]

existing_first_cols = [c for c in first_cols if c in sample_cpdb_summary.columns]
other_cols = [c for c in sample_cpdb_summary.columns if c not in existing_first_cols]

sample_cpdb_summary = sample_cpdb_summary[existing_first_cols + other_cols]

print("样本通讯矩阵维度:", sample_cpdb_summary.shape)
sample_cpdb_summary.head()


In [ ]:
in_file = 'results/cpdb/samples/merged_results/all_samples_cpdb_lr_level_significant.csv'

out_dir = 'results/cpdb/communication'
os.makedirs(out_dir, exist_ok=True)

out_sample_table = os.path.join(
    out_dir,
    "T_samples_celltype_incoming_outgoing_strength_sample_level.csv"
)

out_group_table = os.path.join(
    out_dir,
    "T_samples_celltype_incoming_outgoing_strength_group_summary.csv"
)

out_pdf = os.path.join(
    out_dir,
    "Adult_vs_Child_celltype_incoming_outgoing_strength.pdf"
)

out_png = os.path.join(
    out_dir,
    "Adult_vs_Child_celltype_incoming_outgoing_strength.png"
)

df = pd.read_csv(in_file, low_memory=False)

required_cols = [
    "sample",
    "source",
    "target",
    "interacting_pair",
    "significant_mean",
    "Period_merged",
    "Sample_Type"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if len(missing_cols) > 0:
    raise ValueError(f"缺少必要字段: {missing_cols}")

df["significant_mean"] = pd.to_numeric(
    df["significant_mean"],
    errors="coerce"
)

df = df[df["Sample_Type"].astype(str) == "T"].copy()

print("T样本显著互作表维度:", df.shape)
print("T样本数:", df["sample"].nunique())
print(df["Period_merged"].value_counts(dropna=False))

communication_sample_meta = (
    df[["sample", "Period_merged", "Sample_Type"]]
    .drop_duplicates(subset=["sample"])
    .copy()
)

meta_check = (
    df[["sample", "Period_merged", "Sample_Type"]]
    .drop_duplicates()
    .groupby("sample")
    .size()
)

bad_samples = meta_check[meta_check > 1]
if len(bad_samples) > 0:
    print("以下样本 metadata 不唯一，请检查：")
    print(bad_samples)
    raise ValueError("sample metadata 不唯一")

all_celltypes = sorted(
    list(set(df["source"].dropna().unique()).union(set(df["target"].dropna().unique())))
)

print("细胞类型数量:", len(all_celltypes))
print(all_celltypes)

outgoing_df = (
    df
    .groupby(["sample", "source"], dropna=False)
    .agg(
        outgoing_strength=("significant_mean", "sum"),
        outgoing_lr_count=("interacting_pair", "nunique")
    )
    .reset_index()
    .rename(columns={"source": "celltype"})
)

incoming_df = (
    df
    .groupby(["sample", "target"], dropna=False)
    .agg(
        incoming_strength=("significant_mean", "sum"),
        incoming_lr_count=("interacting_pair", "nunique")
    )
    .reset_index()
    .rename(columns={"target": "celltype"})
)

full_index = pd.MultiIndex.from_product(
    [communication_sample_meta["sample"].unique(), all_celltypes],
    names=["sample", "celltype"]
)

sample_celltype = (
    pd.DataFrame(index=full_index)
    .reset_index()
)

sample_celltype = sample_celltype.merge(
    outgoing_df,
    on=["sample", "celltype"],
    how="left"
)

sample_celltype = sample_celltype.merge(
    incoming_df,
    on=["sample", "celltype"],
    how="left"
)

sample_celltype = sample_celltype.merge(
    communication_sample_meta,
    on="sample",
    how="left"
)

for col in [
    "outgoing_strength",
    "incoming_strength",
    "outgoing_lr_count",
    "incoming_lr_count"
]:
    sample_celltype[col] = sample_celltype[col].fillna(0)

sample_celltype["total_interaction_strength"] = (
    sample_celltype["outgoing_strength"] +
    sample_celltype["incoming_strength"]
)

sample_celltype.to_csv(out_sample_table, index=False)

print("样本层面 incoming/outgoing 表已保存:")
print(out_sample_table)
print(sample_celltype.head())

group_summary = (
    sample_celltype
    .groupby(["Period_merged", "celltype"], dropna=False)
    .agg(
        n_samples=("sample", "nunique"),
        mean_outgoing_strength=("outgoing_strength", "mean"),
        median_outgoing_strength=("outgoing_strength", "median"),
        mean_incoming_strength=("incoming_strength", "mean"),
        median_incoming_strength=("incoming_strength", "median"),
        mean_outgoing_lr_count=("outgoing_lr_count", "mean"),
        mean_incoming_lr_count=("incoming_lr_count", "mean"),
        mean_total_interaction_strength=("total_interaction_strength", "mean")
    )
    .reset_index()
)

group_summary.to_csv(out_group_table, index=False)

print("组内 incoming/outgoing 汇总表已保存:")
print(out_group_table)
print(group_summary.head())

groups = sorted(group_summary["Period_merged"].dropna().unique())
print("Period_merged groups:", groups)

if len(groups) != 2:
    raise ValueError(f"Period_merged 不是两个组，目前为: {groups}")

if "Adult" in groups and "Child" in groups:
    adult_group = "Adult"
    child_group = "Child"
else:
    adult_group = groups[0]
    child_group = groups[1]

plot_groups = [adult_group, child_group]
print("绘图分组顺序:", plot_groups)

color_dict = {
    "GD_T":  "#ffff99",
    "MAI_T": "#c5b0d5",
    "Pro_T": "#fdd0a2",
    "IM_T":  "#98df8a",
    'CD8_TN': '#9ccc65',
    'CD8_t-TEM': '#ffecb3',
    'CD8_TEM': '#ffc107',
    'CD8_TRM': '#8d6e63',
    'CD8_TISG': '#0097a7',
    'CD8_TEMRA': '#ef5350',
    'CD8_TNK': '#e53935',
    'CD8_TEX': '#9c27b0',
    'CD4_TN': '#c8e6c9',
    'CD4_t-TH': '#ffcc80',
    'CD4_TH': '#fb8c00',
    'CD4_TH17': '#ff8a65',
    'CD4_TEMRA': '#e91e63',
    'CD4_TFH': '#ba68c8',
    'CD4_Treg': '#4527a0'
}

for ct in all_celltypes:
    if ct not in color_dict:
        color_dict[ct] = "grey"

all_size_values = group_summary["mean_total_interaction_strength"].fillna(0)

size_min = 60
size_max = 320

size_vmin = all_size_values.min()
size_vmax = all_size_values.max()

def scale_size(v, vmin, vmax):
    if pd.isna(v) or v <= 0:
        return 60
    if vmax == vmin:
        return 120
    return size_min + (v - vmin) / (vmax - vmin) * (size_max - size_min)

xmax = group_summary["mean_outgoing_strength"].max()
ymax = group_summary["mean_incoming_strength"].max()

x_pad = xmax * 0.12 if xmax > 0 else 1
y_pad = ymax * 0.12 if ymax > 0 else 1

xlim = (0, xmax + x_pad)
ylim = (0, ymax + y_pad)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 6),
    sharex=True,
    sharey=True
)

for ax, group in zip(axes, plot_groups):
    sub = group_summary[group_summary["Period_merged"] == group].copy()

    sub["dot_size"] = sub["mean_total_interaction_strength"].apply(
        lambda v: scale_size(v, size_vmin, size_vmax)
    )

    for _, row in sub.iterrows():
        ct = row["celltype"]
        x = row["mean_outgoing_strength"]
        y = row["mean_incoming_strength"]
        s = row["dot_size"]

        ax.scatter(
            x,
            y,
            s=s,
            color=color_dict.get(ct, "grey"),
            edgecolors="white",
            linewidths=0.8,
            alpha=0.9
        )

        ax.text(
            x,
            y,
            ct,
            fontsize=8,
            color=color_dict.get(ct, "black"),
            ha="center",
            va="center"
        )

    ax.set_title(group, fontsize=14, fontstyle="italic")
    ax.set_xlabel("Outgoing interaction strength", fontsize=12, fontstyle="italic")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.5,
        alpha=0.4
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

axes[0].set_ylabel("Incoming interaction strength", fontsize=12, fontstyle="italic")

max_strength = group_summary["mean_total_interaction_strength"].max()

if pd.isna(max_strength) or max_strength <= 0:
    legend_values = [100]
else:
    max_rounded = int(np.ceil(max_strength / 100.0) * 100)

    if max_rounded <= 300:
        legend_values = [100, 200, 300]
    elif max_rounded <= 600:
        legend_values = [200, 400, 600]
    elif max_rounded <= 900:
        legend_values = [300, 600, 900]
    else:
        raw_vals = [max_rounded / 4, max_rounded / 2, max_rounded * 3 / 4]
        legend_values = sorted(
            list(set([int(np.round(v / 100.0) * 100) for v in raw_vals if v > 0]))
        )
        legend_values.append(max_rounded)
        legend_values = sorted(list(set(legend_values)))

legend_values = [v for v in legend_values if v > 0]
print("Legend values:", legend_values)

handles_size = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="lightgrey",
        markeredgecolor="grey",
        markersize=np.sqrt(scale_size(v, size_vmin, size_vmax)) / 1.5,
        label=str(int(v))
    )
    for v in legend_values
]

fig.legend(
    handles=handles_size,
    title="Mean total\ninteraction strength",
    loc="center right",
    bbox_to_anchor=(1.03, 0.55),
    frameon=False,
    fontsize=9,
    title_fontsize=10
)

plt.tight_layout(rect=[0, 0, 0.88, 1])

plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("PDF 已保存:", out_pdf)
print("PNG 已保存:", out_png)


# FigS5f

In [ ]:
IMT_enriched_T=T_all[T_all.obs["Sample_ID"].isin(["HB-P1147-T1-CD45pos-5TB","NB-P979-T1-CD45pos-5TB","GSE147766_NB12"])].copy()


In [ ]:
IMT_enriched_T.obs.index = IMT_enriched_T.obs.index.str.replace("-", "_")

if "T_sub_level_3" not in IMT_enriched_T.obs.columns:
    raise ValueError("data.obs['T_sub_level_3'] 不存在，请确认数据结构是否正确")

counts_file = 'results/cpdb/imt/cellphonedb_counts.h5ad'
counts_data = sc.AnnData(X=IMT_enriched_T.layers["raw.X.log_norm"], obs=pd.DataFrame(index=IMT_enriched_T.obs.index), var=IMT_enriched_T.var)
counts_data.write_h5ad(counts_file)
print(f"Counts 文件已保存: {counts_file}")

meta_file = 'results/cpdb/imt/cellphonedb_meta.csv'
meta_df = IMT_enriched_T.obs[["T_sub_level_3"]].copy()
meta_df.index.name = "Cell"
meta_df.to_csv(meta_file)
print(f"Meta 文件已保存: {meta_file}")


In [ ]:
in_file = 'results/cpdb/imt/cellphonedb_meta.csv'
out_file = 'results/cpdb/imt/cellphonedb_meta_simplified.csv'

meta = pd.read_csv(in_file)

print("原始表格维度:", meta.shape)
print("原始列名:", meta.columns.tolist())
print(meta.head())

celltype_col = meta.columns[1]

print("将处理的细胞类型列:", celltype_col)

meta[celltype_col] = (
    meta[celltype_col]
    .astype(str)
    .str.replace(r"_c\d+", "", regex=True)
)

print("修改后示例:")
print(meta.head())

print("修改后细胞类型数量:", meta[celltype_col].nunique())
print(meta[celltype_col].value_counts().head(20))

meta.to_csv(out_file, index=False)

print("已保存:", out_file)


In [ ]:
cpdb_file_path = 'cellphonedb.zip'
meta_file_path = 'results/cpdb/imt/cellphonedb_meta.csv'
counts_file_path = 'results/cpdb/imt/cellphonedb_counts.h5ad'
output_path= 'results/cpdb/imt/cellphoneDB'
cpdb_results = cpdb_statistical_analysis_method.call(
        cpdb_file_path = cpdb_file_path,
        meta_file_path = meta_file_path,
        counts_file_path = counts_file_path,
        counts_data = 'hgnc_symbol',
        score_interactions = True,
        threshold = 0.1,
        threads = 5,
        output_path = output_path)


# Fig5d&5e&5f&5g

In [ ]:
CD8_T= sc.read_h5ad('CD8_T.h5ad')
CD8_T_tumor=CD8_T[CD8_T.obs["Sample_Type"].isin(["T"])].copy()


In [ ]:
gene = "IKZF2"

if gene not in CD8_T_tumor.var_names:
    raise ValueError(f"{gene} not found in CD8_T_tumor.var_names")

gene_idx = CD8_T_tumor.var_names.get_loc(gene)

expr = CD8_T_tumor.layers["raw.X.log_norm"][:, gene_idx]

CD8_T_tumor.obs[f"{gene}_exp"] = expr.toarray().ravel()


In [ ]:
def summarize_values_by_group(
    adata,
    group_cols,
    value_cols,
    metadata_cols=None,
    value_aggs=("mean", "top10_mean"),
    dropna_group=False,
):

    def p90(x):
        x = pd.Series(x).dropna()
        return np.nan if len(x) == 0 else np.quantile(x, 0.9)

    def top10_mean(x):
        x = pd.Series(x).dropna()
        if len(x) == 0:
            return np.nan
        x = np.sort(x.to_numpy())[::-1]
        k = max(1, int(np.ceil(len(x) * 0.1)))
        return x[:k].mean()

    custom_aggs = {
        "p90": p90,
        "top10_mean": top10_mean,
        "mean": np.mean,
        "median": np.median,
        "sum": np.sum,
        "max": np.max,
        "min": np.min,
        "std": np.std,
    }

    if isinstance(group_cols, str):
        group_cols = [group_cols]
    else:
        group_cols = list(group_cols)

    if isinstance(value_cols, str):
        value_cols = [value_cols]
    else:
        value_cols = list(value_cols)

    if metadata_cols is None:
        metadata_cols = []
    elif isinstance(metadata_cols, str):
        metadata_cols = [metadata_cols]
    else:
        metadata_cols = list(metadata_cols)

    if isinstance(value_aggs, str):
        value_aggs = [value_aggs]
    else:
        value_aggs = list(value_aggs)

    cols = list(dict.fromkeys(group_cols + value_cols + metadata_cols))
    df = adata.obs[cols].copy()

    if dropna_group:
        df = df.dropna(subset=group_cols)

    agg_dict = {}

    agg_dict["n_cells"] = (value_cols[0], "size")

    for v in value_cols:
        for agg in value_aggs:
            func = custom_aggs.get(agg, agg)
            agg_dict[f"{v}_{agg}"] = (v, func)

    for m in metadata_cols:
        agg_dict[m] = (m, "first")

    out = (
        df.groupby(group_cols, dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )

    return out


In [ ]:
def plot_group_box_dots(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    *,
    group_order: Optional[Sequence[str]] = None,
    palette: Optional[Dict[str, str]] = None,
    subgroup_col: Optional[str] = None,
    subgroup_order: Optional[Sequence[str]] = None,
    subgroup_palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,
    figsize: Tuple[float, float] = (6, 4),
    jitter: float = 0.08,
    point_size: float = 20,
    min_box_height_ratio: float = 0.03,
    random_state: int = 0,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    title: Optional[str] = None,
    show_subgroup_legend: bool = True,
    box_edge_color: str = "#C0C0C0",
    box_line_alpha: float = 0.6,
    box_line_width: float = 1.2,
):
    rng = np.random.default_rng(random_state)

    if group_order is None:
        group_order = list(pd.unique(df[group_col].dropna()))
    else:
        group_order = [g for g in group_order if g in df[group_col].values]

    use_cols = [group_col, value_col]
    if subgroup_col is not None:
        use_cols.append(subgroup_col)

    sub = df[use_cols].dropna(subset=[group_col, value_col])
    if sub.empty:
        raise ValueError("筛选后数据为空，请检查 value_col / group_col / subgroup_col。")

    all_vals = sub[value_col].values.astype(float)
    y_min, y_max = float(np.min(all_vals)), float(np.max(all_vals))
    if y_min == y_max:
        y_min -= 0.5
        y_max += 0.5
    y_range = y_max - y_min
    min_box_h = y_range * min_box_height_ratio

    def make_auto_palette(levels, used_colors=None):
        base_tab20 = list(cm.get_cmap("tab20").colors)

        def to_morandi(c, mix=0.85, grey=(0.94, 0.94, 0.94)):
            r, g, b = c[:3]
            return (
                r * mix + grey[0] * (1 - mix),
                g * mix + grey[1] * (1 - mix),
                b * mix + grey[2] * (1 - mix),
            )

        auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if used_colors is not None:
            used_colors = set(c.lower() for c in used_colors)
            auto_colors = [c for c in auto_colors if c.lower() not in used_colors]

        if len(auto_colors) == 0:
            auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if len(auto_colors) < len(levels):
            repeats = int(np.ceil(len(levels) / len(auto_colors)))
            auto_colors = (auto_colors * repeats)[:len(levels)]
        else:
            auto_colors = auto_colors[:len(levels)]

        return dict(zip(levels, auto_colors))

    if palette is None:
        palette = {}
        if meta_color_map is not None:
            for g in group_order:
                if g in meta_color_map:
                    palette[g] = meta_color_map[g]
        for g in group_order:
            if g not in palette:
                palette[g] = box_edge_color

    if subgroup_col is not None:
        if subgroup_order is None:
            sub_levels = list(pd.unique(sub[subgroup_col].dropna()))
        else:
            sub_levels = [s for s in subgroup_order if s in sub[subgroup_col].values]

        final_subgroup_palette = {}

        if meta_color_map is not None:
            for s in sub_levels:
                if s in meta_color_map:
                    final_subgroup_palette[s] = meta_color_map[s]

        if subgroup_palette is not None:
            for s in sub_levels:
                if s in subgroup_palette:
                    final_subgroup_palette[s] = subgroup_palette[s]

        missing_levels = [s for s in sub_levels if s not in final_subgroup_palette]
        if len(missing_levels) > 0:
            auto_pal = make_auto_palette(
                missing_levels,
                used_colors=final_subgroup_palette.values()
            )
            final_subgroup_palette.update(auto_pal)

        subgroup_palette = final_subgroup_palette
    else:
        subgroup_palette = {}
        sub_levels = []

    fig, ax = plt.subplots(figsize=figsize)
    xs = np.arange(len(group_order)) * 0.5
    box_width = 0.30

    whisker_cap_width = box_width / 4
    median_width = box_line_width
    whisker_width = box_line_width
    cap_width = box_line_width
    rect_line_width = box_line_width

    for x, g in zip(xs, group_order):
        g_mask = sub[group_col] == g
        vals = sub.loc[g_mask, value_col].values.astype(float)
        if len(vals) == 0:
            continue

        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        if len(vals) >= 2:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
        else:
            med = vals[0]
            q1 = med - min_box_h / 2
            q3 = med + min_box_h / 2

        if (q3 - q1) < min_box_h:
            center = (q1 + q3) / 2
            q1 = center - min_box_h / 2
            q3 = center + min_box_h / 2

        w_low = float(np.min(vals))
        w_high = float(np.max(vals))

        box_col = palette.get(g, box_edge_color)

        rect = plt.Rectangle(
            (x - box_width / 2, q1),
            box_width,
            q3 - q1,
            facecolor="none",
            edgecolor=box_col,
            linewidth=rect_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )
        ax.add_patch(rect)

        ax.plot(
            [x - box_width / 2, x + box_width / 2],
            [med, med],
            color=box_col,
            linewidth=median_width,
            alpha=box_line_alpha,
            zorder=2,
        )

        ax.plot(
            [x, x], [q3, w_high],
            color=box_col,
            linewidth=whisker_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x, x], [q1, w_low],
            color=box_col,
            linewidth=whisker_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_high, w_high],
            color=box_col,
            linewidth=cap_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_low, w_low],
            color=box_col,
            linewidth=cap_width,
            alpha=box_line_alpha,
            zorder=1
        )

        if subgroup_col is None:
            jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(vals))
            ax.scatter(
                np.full_like(vals, x, dtype=float) + jitter_x,
                vals,
                s=point_size,
                color=box_col,
                alpha=0.85,
                edgecolors="none",
                zorder=3,
            )
        else:
            g_df = sub.loc[g_mask, [value_col, subgroup_col]]
            for sub_name, sdf in g_df.groupby(subgroup_col):
                v = sdf[value_col].values.astype(float)
                v = v[np.isfinite(v)]
                if len(v) == 0:
                    continue

                jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(v))
                col = subgroup_palette.get(sub_name, "#555555")

                ax.scatter(
                    np.full_like(v, x, dtype=float) + jitter_x,
                    v,
                    s=point_size,
                    color=col,
                    alpha=0.9,
                    edgecolors="none",
                    zorder=3,
                    label=sub_name,
                )

    ax.set_xticks(xs)
    ax.set_xticklabels(group_order, rotation=45, ha="right")
    ax.set_ylim(y_min, y_max)
    ax.margins(x=0.05)

    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if title is not None:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if subgroup_col is not None and show_subgroup_legend:
        handles, labels = ax.get_legend_handles_labels()
        handle_by_label = dict(zip(labels, handles))

        if subgroup_order is not None:
            legend_labels = [s for s in subgroup_order if s in handle_by_label]
        else:
            legend_labels = [s for s in sub_levels if s in handle_by_label]

        legend_handles = [handle_by_label[l] for l in legend_labels]

        ax.legend(
            legend_handles,
            legend_labels,
            title=subgroup_col,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
            fontsize="x-small"
        )

    return fig, ax


In [ ]:
CD8_sample_scores = summarize_values_by_group(
    CD8_T_tumor,
    group_cols="Sample_ID",
    value_cols=["NeoTCR8", "IKZF2_exp"],
    metadata_cols=["Disease", "Period_merged"],
    value_aggs=["mean", "top10_mean"]
)
CD8_sample_scores


In [ ]:
disease_order = ['pLGG','pHGG','MB','NB','WT','HB','PB',
                'aHCC','BRCA','CRC','HCC','LC','RC','PC']


In [ ]:
group_col = "Period_merged"
value_col = "NeoTCR8_top10_mean"

fig, ax = plot_group_box_dots(
    df=CD8_sample_scores,
    value_col=value_col,
    group_col=group_col,
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(2, 3.5),
    ylabel="neotcr_top10",
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)
plt.savefig('figures/Fig5d_CD8_NeoTCR8.pdf', dpi=300, bbox_inches="tight")
plt.show()

df = CD8_sample_scores[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

def q(x, q): return float(np.quantile(x, q)) if len(x) else np.nan
summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


In [ ]:
group_col = "Period_merged"
value_col = "IKZF2_exp_mean"
fig, ax = plot_group_box_dots(
    df=CD8_sample_scores,
    value_col=value_col,
    group_col=group_col,
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(2, 3.5),
    ylabel="IKZF2_exp_mean",
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)

plt.show()

df = CD8_sample_scores[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


In [ ]:
def compute_celltype_fraction_per_sample(
    obs_df: pd.DataFrame,
    *,
    sample_id_col: str = "Sample_ID",
    disease_col: str = "Disease",
    celltype_col: str = "anno_TKA"
):

    sample_disease = (
        obs_df[[sample_id_col, disease_col]]
        .dropna(subset=[sample_id_col, disease_col])
        .drop_duplicates(subset=[sample_id_col])
    )

    counts = (
        obs_df
        .groupby([sample_id_col, celltype_col])
        .size()
        .reset_index(name="n")
    )

    totals = (
        counts
        .groupby(sample_id_col)["n"]
        .sum()
        .reset_index(name="total")
    )

    frac_df = counts.merge(totals, on=sample_id_col, how="left")
    frac_df["frac"] = frac_df["n"] / frac_df["total"]

    frac_df = frac_df.merge(sample_disease, on=sample_id_col, how="left")

    frac_df = frac_df[
        [sample_id_col, disease_col, celltype_col, "n", "total", "frac"]
    ]

    count_matrix = (
        counts.pivot(index=sample_id_col,
                     columns=celltype_col,
                     values="n")
        .fillna(0)
        .astype(int)
    )

    count_matrix["total"] = totals.set_index(sample_id_col)["total"]

    count_matrix = count_matrix.join(
        sample_disease.set_index(sample_id_col),
        how="left"
    )

    return frac_df, count_matrix


In [ ]:
CD8_fractions, CD8_counts = compute_celltype_fraction_per_sample(
    CD8_T_tumor.obs,
    sample_id_col="Sample_ID",
    disease_col="Disease",
    celltype_col="anno_CD8T"
)

frac_df_tTEM_c8 = CD8_fractions.loc[
    CD8_fractions["anno_CD8T"] == "CD8_t-TEM_c8"
].copy()

frac_df_tTEM_c8 = frac_df_tTEM_c8.reset_index(drop=True)

CD8_sample_scores = CD8_sample_scores.merge(
    frac_df_tTEM_c8[["Sample_ID", "frac"]].rename(
        columns={"frac": "tTEM_c8_frac"}
    ),
    on="Sample_ID",
    how="left"
)
CD8_sample_scores.head()


In [ ]:
pediatric_color_map = {
    "NB": "#e6550d",
    "HB": "#fd8d3c",
    "WT": "#ffed6f",
    'PB': "#ffffb3",
    "MB": "#e7969c",
    'pLGG': "#d6616b",
    'pHGG': "#ad494a"
}
line_color = "#fdae6b"


In [ ]:
CD8_child_scores = CD8_sample_scores[
    CD8_sample_scores["Period_merged"] == "Child"
].copy()
CD8_child_scores = CD8_child_scores.copy()
CD8_child_scores["Disease"] = CD8_child_scores["Disease"].astype(str)

diseases = CD8_child_scores["Disease"].unique()
palette_filtered = {
    d: pediatric_color_map.get(d, "#999999")
    for d in diseases
}

fig, ax = plt.subplots(figsize=(7,4))

sns.scatterplot(
    data=CD8_child_scores,
    x="IKZF2_exp_mean",
    y="tTEM_c8_frac",
    hue="Disease",
    palette=palette_filtered,
    s=70,
    alpha=0.85,
    edgecolor="none",
    ax=ax,
    legend=False
)

sns.regplot(
    data=CD8_child_scores,
    x="IKZF2_exp_mean",
    y="tTEM_c8_frac",
    scatter=False,
    color="#fdae6b",
    line_kws={"linewidth": 2},
    ax=ax
)

ax.set_xlabel("IKZF2 expression (mean)")
ax.set_ylabel("t-TEM_c8_frac")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

r, p = spearmanr(
    CD8_child_scores["IKZF2_exp_mean"],
    CD8_child_scores["tTEM_c8_frac"]
)

print(f"Spearman r = {r:.3f}, p = {p:.3e}")


In [ ]:
CD8_child_scores = CD8_sample_scores[
    CD8_sample_scores["Period_merged"] == "Child"
].copy()
CD8_child_scores = CD8_child_scores.copy()
CD8_child_scores["Disease"] = CD8_child_scores["Disease"].astype(str)

diseases = CD8_child_scores["Disease"].unique()
palette_filtered = {
    d: pediatric_color_map.get(d, "#999999")
    for d in diseases
}

fig, ax = plt.subplots(figsize=(7,4))

sns.scatterplot(
    data=CD8_child_scores,
    x="IKZF2_exp_mean",
    y="NeoTCR8_top10_mean",
    hue="Disease",
    palette=palette_filtered,
    s=70,
    alpha=0.85,
    edgecolor="none",
    ax=ax,
    legend=False
)

sns.regplot(
    data=CD8_child_scores,
    x="IKZF2_exp_mean",
    y="NeoTCR8_top10_mean",
    scatter=False,
    color="#fdae6b",
    line_kws={"linewidth": 2},
    ax=ax
)

ax.set_xlabel("IKZF2 expression (mean)")
ax.set_ylabel("NeoTCR8_top10_mean")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

r, p = spearmanr(
    CD8_child_scores["IKZF2_exp_mean"],
    CD8_child_scores["NeoTCR8_top10_mean"]
)
print(f"Spearman r = {r:.3f}, p = {p:.3e}")


# FigS5a

In [ ]:
CD4_T= sc.read_h5ad('CD4_T.h5ad')


In [ ]:
CD4_T_tumor=CD4_T[CD4_T.obs["Sample_Type"].isin(["T"])].copy()


In [ ]:
CD4_sample_scores = summarize_values_by_group(
    CD4_T_tumor,
    group_cols="Sample_ID",
    value_cols=["NeoTCR4"],
    metadata_cols=["Disease", "Period_merged"],
    value_aggs=["mean", "top10_mean"]
)
CD4_sample_scores


In [ ]:
group_col = "Period_merged"
value_col = "NeoTCR4_top10_mean"

fig, ax = plot_group_box_dots(
    df=CD4_sample_scores,
    value_col=value_col,
    group_col=group_col,
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(2, 3.5),
    ylabel="neotcr_top10",
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)

plt.show()

df = CD4_sample_scores[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


# FigS5b&c&d

In [ ]:
CD4_nonTreg_tumor = CD4_T_tumor[~CD4_T_tumor.obs["anno_CD4T"].isin(["CD4_Treg_c2","CD4_Treg_c11"])].copy()


In [ ]:
gene = "IKZF2"

if gene not in CD4_nonTreg_tumor.var_names:
    raise ValueError(f"{gene} not found in CD4_nonTreg_tumor.var_names")

gene_idx = CD4_nonTreg_tumor.var_names.get_loc(gene)

expr = CD4_nonTreg_tumor.layers["raw.X.log_norm"][:, gene_idx]

CD4_nonTreg_tumor.obs[f"{gene}_exp"] = expr.toarray().ravel()


In [ ]:
CD4_nonTreg_sample_scores = summarize_values_by_group(
    CD4_nonTreg_tumor,
    group_cols="Sample_ID",
    value_cols=["NeoTCR4", "IKZF2_exp"],
    metadata_cols=["Disease", "Period_merged"],
    value_aggs=["mean", "top10_mean"]
)
CD4_nonTreg_sample_scores


In [ ]:
group_col = "Period_merged"
value_col = "IKZF2_exp_mean"
fig, ax = plot_group_box_dots(
    df=CD4_nonTreg_sample_scores,
    value_col=value_col,
    group_col=group_col,
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(2, 3.5),
    ylabel="IKZF2_exp_mean",
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)
plt.savefig('figures/FigS5b_CD4_nonTreg_IKZF2.pdf', dpi=300, bbox_inches="tight")
plt.show()

df = CD4_nonTreg_sample_scores[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


In [ ]:
CD4_nonTreg_fractions, CD4_nonTreg_counts = compute_celltype_fraction_per_sample(
    CD4_nonTreg_tumor.obs,
    sample_id_col="Sample_ID",
    disease_col="Disease",
    celltype_col="anno_CD4T"
)

frac_df_tTH_c12 = CD4_nonTreg_fractions.loc[
    CD4_nonTreg_fractions["anno_CD4T"] == "CD4_t-TH_c12"
].copy()

frac_df_tTH_c12 = frac_df_tTH_c12.reset_index(drop=True)

CD4_nonTreg_sample_scores = CD4_nonTreg_sample_scores.merge(
    frac_df_tTH_c12[["Sample_ID", "frac"]].rename(
        columns={"frac": "tTH_c12_frac"}
    ),
    on="Sample_ID",
    how="left"
)
CD4_nonTreg_sample_scores.head()


In [ ]:
CD4_nonTreg_child_scores = CD4_nonTreg_sample_scores[
    CD4_nonTreg_sample_scores["Period_merged"] == "Child"
].copy()
CD4_nonTreg_child_scores = CD4_nonTreg_child_scores.copy()
CD4_nonTreg_child_scores["Disease"] = CD4_nonTreg_child_scores["Disease"].astype(str)

diseases = CD4_nonTreg_child_scores["Disease"].unique()
palette_filtered = {
    d: pediatric_color_map.get(d, "#999999")
    for d in diseases
}

fig, ax = plt.subplots(figsize=(7,4))

sns.scatterplot(
    data=CD4_nonTreg_child_scores,
    x="IKZF2_exp_mean",
    y="tTH_c12_frac",
    hue="Disease",
    palette=palette_filtered,
    s=70,
    alpha=0.85,
    edgecolor="none",
    ax=ax,
    legend=False
)

sns.regplot(
    data=CD4_nonTreg_child_scores,
    x="IKZF2_exp_mean",
    y="tTH_c12_frac",
    scatter=False,
    color="#fdae6b",
    line_kws={"linewidth": 2},
    ax=ax
)

ax.set_xlabel("IKZF2 expression (mean)")
ax.set_ylabel("t-TH_c12_frac")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig('figures/FigS5c_CD4_nonTreg_IKZF2_tTH.pdf', bbox_inches="tight")
plt.show()

r, p = spearmanr(
    CD4_nonTreg_child_scores["IKZF2_exp_mean"],
    CD4_nonTreg_child_scores["tTH_c12_frac"]
)

print(f"Spearman r = {r:.3f}, p = {p:.3e}")


In [ ]:
CD4_nonTreg_child_scores = CD4_nonTreg_sample_scores[
    CD4_nonTreg_sample_scores["Period_merged"] == "Child"
].copy()
CD4_nonTreg_child_scores = CD4_nonTreg_child_scores.copy()
CD4_nonTreg_child_scores["Disease"] = CD4_nonTreg_child_scores["Disease"].astype(str)

diseases = CD4_nonTreg_child_scores["Disease"].unique()
palette_filtered = {
    d: pediatric_color_map.get(d, "#999999")
    for d in diseases
}

fig, ax = plt.subplots(figsize=(7,4))

sns.scatterplot(
    data=CD4_nonTreg_child_scores,
    x="IKZF2_exp_mean",
    y="NeoTCR4_top10_mean",
    hue="Disease",
    palette=palette_filtered,
    s=70,
    alpha=0.85,
    edgecolor="none",
    ax=ax,
    legend=False
)

sns.regplot(
    data=CD4_nonTreg_child_scores,
    x="IKZF2_exp_mean",
    y="NeoTCR4_top10_mean",
    scatter=False,
    color="#fdae6b",
    line_kws={"linewidth": 2},
    ax=ax
)

ax.set_xlabel("IKZF2 expression (mean)")
ax.set_ylabel("NeoTCR4_top10_mean")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig('figures/FigS5d_CD4_nonTreg_IKZF2_NeoTCR4.pdf', bbox_inches="tight")
plt.show()

r, p = spearmanr(
    CD4_nonTreg_child_scores["IKZF2_exp_mean"],
    CD4_nonTreg_child_scores["NeoTCR4_top10_mean"]
)
print(f"Spearman r = {r:.3f}, p = {p:.3e}")
